# DataBench 2: one fixed question, the newest decision models

Round two of the call-labeling benchmark (`02_benchmark.ipynb` is round one and stays frozen as published). Same 1,000 ABCD test conversations, same scorers, same rules, with two changes:

1. **One question is redefined.** Round one's biggest error was definitional: the identity question asked whether the agent "verified the customer's identity using their details", the truth is the agent running the `verify-identity` action, and every LLM read a name lookup as verification (378-471 false alarms each). The new wording describes the logged event precisely. It is iterated on a **dev** sample only, frozen, then scored on test once. The other six questions keep round one's exact wording.
2. **The newest models.** Jev 1.14 joins Jev 1.13; Cloudflare's Clef and Clef-flash join; and because Clef-flash is open weights (Apache 2.0), it runs on **three hosts**: Cloudflare Workers AI, GreenPT's Metis (Clef-flash at full precision on EU renewable power) and a local RTX 5070 Ti. The cheap LLMs are unchanged: Haiku 5.5, GPT-6 Luna and Gemini 3.5 Flash-Lite are still each provider's newest budget model.

| Family | Models |
|---|---|
| Keyword rules / supervised ML | one regex per label; TF-IDF + logistic regression (baselines, as in round one) |
| Cheap LLMs | Claude Haiku 5.5, GPT-6 Luna, Gemini 3.5 Flash-Lite |
| System One (decision models) | Jev 1.13, Jev 1.14, Clef, Clef-flash on Cloudflare, Metis (GreenPT), Clef-flash on the local GPU |

Dropped from round one: Claude Haiku 4.5 (retired: Haiku 5.5 does the same job at 1/20 of the cost) and Laya (near chance untuned, by its own docs; it returns only fine-tuned). The learning curve isn't repeated; round one has it.

**Questions this round answers**
- Did one sentence fix the identity question, for every model? (round one vs round two answers, paired, same chats)
- Did changing one question move any of the other six answers? (the collateral check)
- Did the new decision models change the leaderboard, and did Jev 1.14 improve on 1.13?
- Does where you run the same weights change the answers? (Clef-flash on three hosts)
- How many probe flips are just run-to-run noise? (the repeat control round one lacked)
- Two suggestions from readers of round one, tested rather than assumed: avisangle's rewording ("did the agent confirm the details matched the account", plus one example of what isn't verification) is wording `v2`; xapep's check (make the model quote the line it bases "verified" on, and distrust a quote that isn't in the transcript) is the **evidence arm** in section 13b. xapep's "false yeses per dollar" is a column in section 16, and the exact prompts of every version are exported to `harness/` (section 7b).

**Sections:** 1 Load, 2 Questions, 3 Scorers, 4 Keyword baseline, 5 Supervised baseline, 6 Prompt, 7 Models, 8 Smoke test, 9 Run machinery, 10 Dev: the identity wording, 11 Test run, 12 Results, 13 Before and after the fix, 14 Same weights, three hosts, 15 Memorization probe and repeat control, 16 Cost and energy at scale, 17 Precision and recall (thresholds chosen on dev), Export, Figures, Caveats.

## 1. Load

Same pinned download as `01_explore.ipynb`, cached outside the repo.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib typesafe-sdk anthropic openai google-genai requests huggingface_hub nvidia-ml-py

import gzip
import json
import os
import random
import re
import time
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ABCD_COMMIT = "6b8700ce67c6b37b062dd7a60abc76d7ef832a97"   # asappresearch/abcd, pinned
ABCD_FILES = ["data/abcd_v1.1.json.gz", "data/kb.json", "data/ontology.json"]
CACHE_DIR = Path.home() / ".cache" / "abcd" / ABCD_COMMIT
for file in ABCD_FILES:
    local_path = CACHE_DIR / file
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/asappresearch/abcd/{ABCD_COMMIT}/{file}", local_path)

with gzip.open(CACHE_DIR / "data/abcd_v1.1.json.gz") as f:
    abcd = json.load(f)
policy_actions = json.loads((CACHE_DIR / "data/kb.json").read_text())   # the 55 intents that have an action policy
print({split: len(conversations) for split, conversations in abcd.items()})

## 2. Questions and ground truth

`QUESTIONS` is the single source of truth: the wording and option descriptions every model sees. Option descriptions were written from the label names and the dataset's own ontology, before any model was run.

- **reason** (choice, 10): the scenario's `flow`.
- **intent** (choice, 55): the dataset's official 55 intents. For the two FAQ flows the scenario's `subflow` is a specific FAQ article (`boots_how_2`, `timing_1`); it maps to its intent by the name before the article number (`boots`, `timing`), as in the ABCD ontology.
- **four yes/no questions**: whether the matching action appears in the system log.
- **resolution** (score, 3 ordered levels) from the log: money back (refund or promo code) > changed something (order, account, password, purchase) > information only.

**The one change: `identity_verified`.** The truth is the `verify-identity` action. ABCD's own ontology shows what separates it from the two look-alikes that fooled round one's models:

| Action | Values the agent enters | What it is |
|---|---|---|
| `verify-identity` | always **3**: full name plus two of account ID, order ID, zip code, phone | the truth for this question |
| `pull-up-account` | **1**: a name or an account ID | account lookup, not verification |
| `validate-purchase` | username, email and order ID | checks a purchase, not the person |

366 of the 1,000 test conversations have a lookup or a purchase validation but no `verify-identity`. `IDENTITY_WORDINGS` keeps round one's wording as `v0` (for the dev comparison) and each rewrite as `v1`, `v2`, ... (at most three). `FROZEN_IDENTITY_VERSION` is set by hand after section 10 and before the test run; the run refuses to start without it.

In [ ]:
FLOW_DESCRIPTIONS = {
    "account_access": "Can't get into their account: forgot username or password, or two-factor authentication problems",
    "manage_account": "Wants to change account details or services: address, name, phone, payment method, added or removed services",
    "order_issue": "Problem with an existing order: unexpected fee, delivery time, payment method, quantity, or wants to change or cancel the order",
    "product_defect": "Received a faulty or wrong item and wants a return or refund, or asks about a refund",
    "purchase_dispute": "Disputes a charge or price: price dropped, competitor is cheaper, item out of stock, promo code failed, billed for something not bought or already returned",
    "shipping_issue": "Shipping problem: package status, change shipping details, missing package, or shipping cost",
    "single_item_query": "General question about one product (how it works, materials, sizing) with no order problem",
    "storewide_query": "General question about store-wide policies: pricing, timing, membership, or return policy",
    "subscription_inquiry": "Question about their subscription or membership: status, due date or amount, paying the bill, or extending it",
    "troubleshoot_site": "The website isn't working: slow, search results wrong, shopping cart or credit card not working",
}
FAQ_FLOWS = {"single_item_query", "storewide_query"}

def readable(name):
    return name.replace("_", " ").replace("2fa", "two-factor authentication")

# Intent options: ABCD's 55 official intents, each described as "<flow>: <intent>" straight from the ontology names
ontology = json.loads((CACHE_DIR / "data/ontology.json").read_text())
INTENT_FLOW = {intent: flow for flow, intents in ontology["intents"]["subflows"].items() for intent in intents}
INTENT_DESCRIPTIONS = {intent: f"{readable(flow)}: {readable(intent)}" for intent, flow in sorted(INTENT_FLOW.items())}

RESOLUTION_LEVELS = {   # ordered, lowest first; the index is the score
    "information_only": "Information only: the agent answered questions, looked things up or passed the issue on, but changed nothing and gave nothing",
    "changed_something": "Changed something: the agent updated an order or account, reset a password, or placed a purchase, but gave no money back",
    "money_back": "Money back: the agent issued a refund or gave the customer a promo code",
}
MONEY_ACTIONS = {"offer-refund", "promo-code"}
CHANGE_ACTIONS = {"update-order", "update-account", "make-password", "make-purchase"}

# Round two changes ONE question. v0 is round one's wording (kept for the dev comparison); v1+ are rewrites,
# written from the log's action definitions (ontology.json) and iterated on the dev sample only (section 10).
IDENTITY_WORDINGS = {
    "v0": {"instructions": "Did the agent verify the customer's identity using their details (for example name, account ID, order ID or zip code)?",
           "criteria": {"true": "The agent checked the customer's identity", "false": "No identity check was done"}},
    "v1": {"instructions": ("Did the agent run the store's formal identity verification, checking three of the customer's details "
                            "together: their full name plus two of account ID, order ID, zip code or phone number?"),
           "criteria": {"true": "Formal identity verification was done: three details checked together",
                        "false": ("No formal verification. Pulling up the account from only a name or account ID does not count, "
                                  "and neither does validating a purchase with username, email and order ID")}},
    # v2: a reader's suggestion (avisangle, Reddit), as written, plus one example of what is NOT verification
    "v2": {"instructions": "Did the agent confirm that the customer's details matched the account?",
           "criteria": {"true": "The agent confirmed the customer's details matched the account",
                        "false": ("The agent did not confirm a match. For example, an agent who asks 'May I have your full name?' "
                                  "and then answers a general question has asked for details, not verified them")}},
    # v3: the store's own policy (kb.json) plus the clues that separate verified chats on the TRAIN split
    # (zip code, phone, account ID with order ID). Written from train and kb.json only, never from dev or test.
    "v3": {"instructions": ("Did the agent verify the customer's identity? In this store, agents verify identity before changing an "
                            "account, answering subscription or billing questions, checking or changing an order, handling a price "
                            "dispute, or recovering a username, usually by asking for a zip code, a phone number, or an account ID "
                            "together with an order ID. For refunds, returns, missing items, promo codes and password resets they "
                            "only look the account up by name or account ID, which is not verification."),
           "criteria": {"true": "The agent verified the customer's identity",
                        "false": "No verification: the agent only looked the account up, asked for details without verifying, or needed no account"}},
}
FROZEN_IDENTITY_VERSION = "v3"   # frozen 2026-10-20 on dev (section 10), before any test call; never edited afterwards

QUESTIONS = {
    "reason": {"type": "choice", "instructions": "Why did the customer contact the store?", "criteria": FLOW_DESCRIPTIONS},
    "intent": {"type": "choice", "instructions": "What specific request did the customer make?", "criteria": INTENT_DESCRIPTIONS},
    "identity_verified": {"type": "noul", **IDENTITY_WORDINGS[FROZEN_IDENTITY_VERSION or "v1"]},   # rebuilt by use_identity_wording()
    "refund_offered": {"type": "noul", "instructions": "Did the agent issue a refund to the customer?",
                       "criteria": {"true": "A refund was issued", "false": "No refund was issued (refunds may be mentioned, asked about or denied)"}},
    "promo_given": {"type": "noul", "instructions": "Did the agent give the customer a promo code?",
                    "criteria": {"true": "A promo code was given", "false": "No promo code was given"}},
    "handed_off": {"type": "noul", "instructions": "Did the agent pass the issue to another team (a manager, the purchasing department or the website team)?",
                   "criteria": {"true": "The issue was passed to another team", "false": "The agent handled it without passing it on"}},
    "resolution": {"type": "score", "instructions": "How far did the agent go for the customer?", "criteria": RESOLUTION_LEVELS},
}
NOUL_ACTIONS = {"identity_verified": "verify-identity", "refund_offered": "offer-refund",
                "promo_given": "promo-code", "handed_off": "notify-team"}
CHOICE_QUESTIONS = [name for name, question in QUESTIONS.items() if question["type"] == "choice"]
NOUL_QUESTIONS = [name for name, question in QUESTIONS.items() if question["type"] == "noul"]
SCORE_QUESTIONS = [name for name, question in QUESTIONS.items() if question["type"] == "score"]
print(f"{len(INTENT_DESCRIPTIONS)} intent options")

In [ ]:
def conversation_text(conversation):
    # Agent and customer lines only: action lines are the ground truth and never reach a model
    return "\n".join(f"{speaker.capitalize()}: {text}" for speaker, text in conversation["original"] if speaker != "action")

def ground_truth(conversation):
    actions = {turn["targets"][2] for turn in conversation["delexed"] if turn["speaker"] == "action"}
    flow, subflow = conversation["scenario"]["flow"], conversation["scenario"]["subflow"]
    labels = {"reason": flow, "intent": subflow.split("_")[0] if flow in FAQ_FLOWS else subflow}   # FAQ article -> its intent
    labels.update({question: action in actions for question, action in NOUL_ACTIONS.items()})
    labels["resolution"] = 2 if actions & MONEY_ACTIONS else 1 if actions & CHANGE_ACTIONS else 0
    return labels

def build_split(split):
    rows = [{"convo_id": conversation["convo_id"], "text": conversation_text(conversation), **ground_truth(conversation)}
            for conversation in abcd[split]]
    frame = pd.DataFrame(rows).sort_values("convo_id").reset_index(drop=True)   # fixed order for every model
    # 33 conversations (4 in test) carry an intent missing from ABCD's own ontology (`status_questions`, `status_delivery_date`): no right option exists
    off_ontology = ~frame["intent"].isin(INTENT_DESCRIPTIONS)
    print(f"{split}: dropped {off_ontology.sum()} conversation(s) with an off-ontology intent: {sorted(frame.loc[off_ontology, 'intent'])}")
    return frame[~off_ontology].reset_index(drop=True)

train, dev, test = build_split("train"), build_split("dev"), build_split("test")
conversation_by_id = {conversation["convo_id"]: conversation for conversations in abcd.values() for conversation in conversations}

label_summary = {}
for question in NOUL_QUESTIONS:
    label_summary[question] = {"train positives": int(train[question].sum()), "test positives": int(test[question].sum()),
                               "test share": round(test[question].mean(), 3)}
display(pd.DataFrame(label_summary).T)
print("resolution levels in test:", test["resolution"].map(dict(enumerate(RESOLUTION_LEVELS))).value_counts().to_dict())

## 3. Scorers

One scorer per answer type, shared by every model. Bootstrap 95% intervals use the **same resamples for every model** (fixed seed), so intervals are directly comparable.

- **choice**: accuracy (headline) and macro-F1 (every option counts equally).
- **noul**: F2 on "yes" (rewards catching the rare positives, still punishes flagging everything), plus recall, precision and, for models that return a probability, AUC.
- **score**: exact accuracy and quadratic weighted kappa (partial credit for landing one level off).

**Invalid answers are always scored as wrong**, never dropped: a missing choice becomes `<invalid>`, a missing yes/no becomes the opposite of the truth, and a missing level becomes the level farthest from the truth. Each model's share of valid answers is reported next to its scores.

**A perfect score has a degenerate interval.** When a model gets every case right, every bootstrap resample is also perfect, so the interval is 1.00-1.00. That says the test set holds no counterexample, not that the model can't miss one; with 52 positives, a true miss rate of a few percent could easily go unseen.

In [ ]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, roc_auc_score

BOOTSTRAP_RESAMPLES = 2000
RESAMPLE_INDEX = np.random.default_rng(SEED).integers(0, len(test), size=(BOOTSTRAP_RESAMPLES, len(test)))

def bootstrap_ci(metric, true_values, predicted_values):
    true_values, predicted_values = np.asarray(true_values), np.asarray(predicted_values)
    samples = [metric(true_values[index], predicted_values[index]) for index in RESAMPLE_INDEX]
    low, high = np.percentile(samples, [2.5, 97.5])
    return f"{low:.2f}-{high:.2f}"

def f2(true_values, predicted_values):
    tp = np.sum(true_values & predicted_values)
    return 5 * tp / max(5 * tp + 4 * np.sum(true_values & ~predicted_values) + np.sum(~true_values & predicted_values), 1)

def score_question(question, true_values, predicted_values, probabilities=None):
    question_type = QUESTIONS[question]["type"]
    if question_type == "choice":
        return {"accuracy": accuracy_score(true_values, predicted_values),
                "95% CI": bootstrap_ci(accuracy_score, true_values, predicted_values),
                "macro F1": f1_score(true_values, predicted_values, average="macro", zero_division=0)}
    if question_type == "noul":
        true_values, predicted_values = np.asarray(true_values, bool), np.asarray(predicted_values, bool)
        tp = int(np.sum(true_values & predicted_values))
        result = {"F2": f2(true_values, predicted_values), "95% CI": bootstrap_ci(f2, true_values, predicted_values),
                  "recall": tp / max(true_values.sum(), 1), "precision": tp / max(predicted_values.sum(), 1),
                  "TP": tp, "FN": int(np.sum(true_values & ~predicted_values)), "FP": int(np.sum(~true_values & predicted_values))}
        if probabilities is not None:
            result["AUC"] = roc_auc_score(true_values, probabilities)
        return result
    quadratic_kappa = lambda a, b: cohen_kappa_score(a, b, weights="quadratic", labels=list(range(len(RESOLUTION_LEVELS))))
    return {"accuracy": accuracy_score(true_values, predicted_values),
            "95% CI": bootstrap_ci(accuracy_score, true_values, predicted_values),
            "weighted kappa": quadratic_kappa(true_values, predicted_values)}

all_predictions, all_probabilities = {}, {}   # model -> question -> array over test rows

def score_model(model_name):
    return {question: score_question(question, test[question].values, all_predictions[model_name][question],
                                     all_probabilities.get(model_name, {}).get(question))
            for question in QUESTIONS}

## 4. Keyword baseline

The zero-training floor: one regular expression per yes/no label, written from the label names. `01_explore.ipynb` showed these keywords already separate the labels well, so any model has to beat `grep` to earn credit. There is no keyword version of the two choice questions (55 intents don't reduce to one regex each), so it's scored on the yes/no and resolution questions only.

In [ ]:
KEYWORDS = {
    "identity_verified": r"verif|zip ?code|account id",
    "refund_offered": r"\brefund",
    "promo_given": r"\bpromo|\bcode\b|discount",
    "handed_off": r"manager|supervisor|department|team|escalat",
}
CHANGE_KEYWORDS = r"\bupdat|\bchang|\bcancel|\breset|new password|\border(?:ed)? (?:it|that|one)"

def keyword_predictions(frame):
    predictions = {question: frame["text"].str.contains(pattern, flags=re.IGNORECASE, regex=True).values
                   for question, pattern in KEYWORDS.items()}
    money = predictions["refund_offered"] | predictions["promo_given"]
    changed = frame["text"].str.contains(CHANGE_KEYWORDS, flags=re.IGNORECASE, regex=True).values
    predictions["resolution"] = np.where(money, 2, np.where(changed, 1, 0))
    return predictions

all_predictions["Keywords"] = keyword_predictions(test)
keyword_scores = {question: score_question(question, test[question].values, prediction)
                  for question, prediction in all_predictions["Keywords"].items()}
pd.DataFrame(keyword_scores).T

## 5. Supervised baseline

TF-IDF (words and word pairs) + logistic regression, one model per question, trained on the full train split, neutral settings, exactly as in round one. It's the reference for what ~8,000 hand labels buy. The learning curve isn't repeated here: round one's `02_benchmark.ipynb` has it, and the training data and model are unchanged.

The supervised model and the keyword rules are trained on the dataset's labels, not on question wording, so the identity change doesn't touch them.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

def train_tfidf(training_rows, question):
    model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                          LogisticRegression(max_iter=2000, C=4.0))
    return model.fit(training_rows["text"], training_rows[question])

def headline_metric(question, true_values, predicted_values):
    kind = QUESTIONS[question]["type"]
    if kind == "noul":
        return f2(np.asarray(true_values, bool), np.asarray(predicted_values, bool))
    return accuracy_score(true_values, predicted_values)

SUPERVISED = f"TF-IDF + LR ({len(train):,} labels)"
all_predictions[SUPERVISED], all_probabilities[SUPERVISED] = {}, {}
for question in QUESTIONS:
    model = train_tfidf(train, question)
    all_predictions[SUPERVISED][question] = model.predict(test["text"])
    if QUESTIONS[question]["type"] == "noul":
        all_probabilities[SUPERVISED][question] = model.predict_proba(test["text"])[:, 1]
pd.DataFrame(score_model(SUPERVISED)).T

## 6. One prompt, three request shapes

Everything is rendered from `QUESTIONS`:
- **Jev / Laya**: the transcript is `state`; each question is a typed `choice` / `noul` / `score` question with the same instructions and option descriptions.
- **LLMs**: the same instructions and descriptions as text, the transcript last (so the identical question block can be cached), and a JSON schema that only allows the listed options.

For score answers, Jev and Laya return a distribution over levels. Their docs warn that the expected value is unreliable as a magnitude, so the **most likely level** is used.

In [ ]:
LLM_TASK = ("You are labeling a customer-service conversation between an agent and a customer of an online clothing store. "
            "Answer every question using only the conversation. Reply with a JSON object that has one key per question, "
            "using exactly one of the listed option keys as each value.")

def render_question_block():
    lines = []
    for name, question in QUESTIONS.items():
        lines.append(f"### {name}\n{question['instructions']}")
        if question["type"] == "noul":
            lines.append(f"- yes: {question['criteria']['true']}\n- no: {question['criteria']['false']}")
        else:
            lines.extend(f"- {key}: {description}" for key, description in question["criteria"].items())
        lines.append("")
    return "\n".join(lines)

LLM_QUESTION_BLOCK = f"{LLM_TASK}\n\n## Questions\n\n{render_question_block()}"

def llm_option_keys(name):
    question = QUESTIONS[name]
    return ["yes", "no"] if question["type"] == "noul" else list(question["criteria"])

LLM_SCHEMA = {
    "type": "object",
    "properties": {name: {"type": "string", "enum": llm_option_keys(name)} for name in QUESTIONS},
    "required": list(QUESTIONS),
    "additionalProperties": False,
}

LLM_CONVERSATION_TEMPLATE = "## Conversation\n{transcript}\n"

def llm_prompt_parts(transcript):
    return LLM_QUESTION_BLOCK, LLM_CONVERSATION_TEMPLATE.format(transcript=transcript)   # (identical cacheable prefix, per-call suffix)

def parse_llm_answers(raw_text):
    # Every answer checked against its option list; anything else -> None (invalid, scored as wrong)
    try:
        answers = json.loads(raw_text)
    except (json.JSONDecodeError, TypeError):
        return {name: None for name in QUESTIONS}
    parsed = {}
    for name, question in QUESTIONS.items():
        value = answers.get(name) if isinstance(answers, dict) else None
        if value not in llm_option_keys(name):
            parsed[name] = None
        elif question["type"] == "noul":
            parsed[name] = value == "yes"
        elif question["type"] == "score":
            parsed[name] = list(question["criteria"]).index(value)
        else:
            parsed[name] = value
    return parsed

def system_one_questions():
    # Plain JSON in Jev's wire format; Clef (Workers AI), Metis and the local release loader take the same shape
    rendered = {}
    for name, question in QUESTIONS.items():
        if question["type"] == "score":
            rendered[name] = {"type": "score", "instructions": question["instructions"], "criteria": list(question["criteria"].values())}
        else:
            rendered[name] = {"type": question["type"], "instructions": question["instructions"], "criteria": question["criteria"]}
    return rendered

SYSTEM_ONE_QUESTIONS = system_one_questions()

def most_likely_level(level_probabilities, question):
    # Jev keys levels by index ("0", "1", ...). Tolerate a list, or keys that are the level descriptions, from the REST
    # hosts, whose exact score format the smoke test confirms; an unrecognised key raises and the call is logged as malformed.
    if isinstance(level_probabilities, list):
        level_probabilities = dict(enumerate(level_probabilities))
    descriptions = list(question["criteria"].values())
    by_index = {}
    for level, p in level_probabilities.items():
        try:
            by_index[int(level)] = p
        except (TypeError, ValueError):
            by_index[descriptions.index(level)] = p
    return max(by_index, key=by_index.get)

def parse_system_one_answers(answers):
    # answers: name -> {"choice"/"noul"/"score", "probabilities"}; returns (labels, probabilities for yes/no)
    labels, probabilities = {}, {}
    for name, question in QUESTIONS.items():
        answer = answers[name]
        if question["type"] == "choice":
            labels[name] = answer["choice"]
        elif question["type"] == "noul":
            probabilities[name] = float(answer["noul"])
            labels[name] = probabilities[name] >= 0.5
        else:
            labels[name] = most_likely_level(answer["probabilities"], question)
    return labels, probabilities

print(LLM_QUESTION_BLOCK[:1500], "...")
print(f"\nquestion block: {len(LLM_QUESTION_BLOCK):,} chars")
print("parser check:", parse_llm_answers('{"reason": "shipping_issue"}')["reason"], parse_llm_answers("not json")["reason"])

## 7. The models

| Model | Host | Model ID | Price $/1M tokens | Notes |
|---|---|---|---|---|
| Claude Haiku 5.5 | Anthropic | `claude-haiku-5-5` | 0.10 in / 0.01 cached / 0.125 cache write / 0.50 out | provider-default reasoning (adaptive, effort medium), as in round one |
| GPT-6 Luna | OpenAI | `gpt-6-luna` | 0.10 / 0.01 / 0.125 / 0.50 | as in round one |
| Gemini 3.5 Flash-Lite | Google | `gemini-3.5-flash-lite` | 0.30 / 0.03 / none / 2.50 | as in round one |
| Jev 1.13 | TypeSafe | `jev-1.13.0` | 0.042 input only | round one's Jev, re-run on the new wording so 1.13 vs 1.14 compares like with like |
| Jev 1.14 | TypeSafe | `jev-1.14.0` | 0.042 input only (unchanged) | released 2026-10-14; pinned (`jev-latest` moves to it about 2026-10-21); experimental "danger zone" features left off |
| Clef | Cloudflare Workers AI | `@cf/cloudflare/clef` | 0.24 input only | 27B, open weights, released 2026-10-01 |
| Clef-flash (Cloudflare) | Cloudflare Workers AI | `@cf/cloudflare/clef-flash` | 0.038 input only | 9B |
| Metis (GreenPT) | GreenPT, EU only | `metis` | €0.12 input only (≈ $0.135 at the ECB rate) | Clef-flash at full precision, per GreenPT's docs; reports energy and emissions per call |
| Clef-flash (local) | RTX 5070 Ti | `Cloudflare/clef-flash` at a pinned revision | $0 (hardware and power not counted) | the card's release loader, quantized to fit: BF16 needs ~18-21 GB |

- **Prices** for Haiku 5.5, Luna and Flash-Lite are round one's, from each provider's pricing page; Jev's is from TypeSafe (unchanged with 1.14). Clef ($0.240) and Clef-flash ($0.038, after Cloudflare's cut) are from the Workers AI pricing page (updated and checked 2026-10-19), which bills $0.011 per 1,000 neurons; the 10,000 free neurons a day aren't subtracted. Metis's price comes from GreenPT's `GET /v1/pricing/metis` (`"promptToken": 0.12`, checked 2026-10-19); GreenPT's docs quote API prices in EUR per 1M tokens, converted at the ECB reference rate of 2026-10-19 (1.1225 USD per EUR). Cost is always recomputed from logged tokens and the current price table, so fixing a price later fixes every number.
- **Configuration fingerprints are per provider now.** Round one hashed every provider's settings into every row, so changing, say, a local-model setting would have invalidated every API row. Here each row's fingerprint covers the prompt, the identity wording version, the model's own settings, the shared request settings and only its own provider's settings. Timeouts and request pacing (`TRANSPORT`) are not fingerprinted: they change how long the notebook waits, never what a model is asked.
- **Metis is EU-only.** GreenPT's docs say the US endpoint returns `403 endpoint_not_available_in_location`; the EU endpoint did answer from the US in the smoke test on 2026-10-19 (Clef, Clef-flash and Metis all parsed cleanly, all reported 2,684 input tokens). GreenPT also says the model behind `metis` can change without the ID changing, so every call logs the `impact` report (with its `version`) and the served model, and section 12 checks they never changed during the run.
- **The local model** loads through the model card's release loader (`load_release_model` and `systemone` from `joint_schema_model.py`). That file is code shipped with the download: pin the revision and read the file at that revision before the first load. GPU energy per call is read from the driver's energy counter (NVML) around each call: whole-GPU energy, not the model alone.
- **No tools or web search** for any model; OpenAI calls use `store=False`.

In [ ]:
import csv
import hashlib
import sys
from dataclasses import asdict, dataclass, fields
from datetime import datetime, timezone

import anthropic
import openai
import requests
from google import genai
from google.genai import types as genai_types
from typesafe_sdk import TypeSafeClient

# Shared request settings: in every row's fingerprint
REQUEST_SETTINGS = {"max_output_tokens": 16_000}
# Per-provider settings: each row's fingerprint covers only its own provider's entry
PROVIDER_SETTINGS = {
    "anthropic": {"cache_control": {"type": "ephemeral"}},                  # on the question block
    "openai": {"cache_breakpoint": {"mode": "explicit"}, "prompt_cache_key": "abcd-call-labeling", "store": False},
    "google": {"joiner": "\n"},                                             # Gemini takes one string: block + joiner + conversation
    "typesafe": {},
    "cloudflare": {"url": "https://api.cloudflare.com/client/v4/accounts/{account}/ai/run/{model}"},
    "greenpt": {"url": "https://api.eu.greenpt.ai/v1/systemone"},           # Metis is EU-only; the US endpoint answers 403
    "clef_local": {"revision": None,      # a Hugging Face commit hash: required before the local model loads
                   "load_kwargs": {},     # BF16 needs ~18-21 GB, 16 GB here: set the loader's quantization option after reading it
                   "device": "cuda"},
}
# Not fingerprinted: how long to wait, never what is asked
TRANSPORT = {"http_timeout_s": 60, "typesafe_timeout_s": 60,
             "min_interval_s": {"greenpt": 0.55,     # GreenPT: 600 requests per 5 minutes, shared across its endpoints
                                "cloudflare": 3.1}}  # Workers AI answered 429 at ~40/min; its paid-model limit is 20/min per model
EUR_TO_USD = 1.1225   # ECB reference rate, 2026-10-19 (GreenPT prices in EUR)

# provider picks the caller. reasoning: the setting passed, or None = parameter omitted (provider default).
MODELS = {
    "Claude Haiku 5.5":        {"provider": "anthropic", "model_id": "claude-haiku-5-5", "key_env": "CLAUDE_KEY", "reasoning": None,
                                "price": {"input": 0.10, "cached_input": 0.01, "cache_write": 0.125, "output": 0.50}},
    "GPT-6 Luna":              {"provider": "openai", "model_id": "gpt-6-luna", "key_env": "GPT_KEY", "reasoning": None,
                                "price": {"input": 0.10, "cached_input": 0.01, "cache_write": 0.125, "output": 0.50}},
    "Gemini 3.5 Flash-Lite":   {"provider": "google", "model_id": "gemini-3.5-flash-lite", "key_env": "GEM_KEY", "reasoning": None,
                                "price": {"input": 0.30, "cached_input": 0.03, "cache_write": 0.00, "output": 2.50}},
    "Jev 1.13":                {"provider": "typesafe", "model_id": "jev-1.13.0", "key_env": "JEV_KEY", "reasoning": None,
                                "price": {"input": 0.042, "cached_input": 0.042, "cache_write": 0.00, "output": 0.00}},
    "Jev 1.14":                {"provider": "typesafe", "model_id": "jev-1.14.0", "key_env": "JEV_KEY", "reasoning": None,
                                "price": {"input": 0.042, "cached_input": 0.042, "cache_write": 0.00, "output": 0.00}},
    "Clef":                    {"provider": "cloudflare", "model_id": "@cf/cloudflare/clef", "key_env": "CLEF_KEY", "reasoning": None,
                                "price": {"input": 0.24, "cached_input": 0.24, "cache_write": 0.00, "output": 0.00}},
    "Clef-flash (Cloudflare)": {"provider": "cloudflare", "model_id": "@cf/cloudflare/clef-flash", "key_env": "CLEF_KEY", "reasoning": None,
                                "price": {"input": 0.038, "cached_input": 0.038, "cache_write": 0.00, "output": 0.00}}, # Workers AI page, 2026-10-19
    "Metis (GreenPT)":         {"provider": "greenpt", "model_id": "metis", "key_env": "METIS_KEY", "reasoning": None,
                                "price": {"input": 0.12 * EUR_TO_USD, "cached_input": 0.12 * EUR_TO_USD, "cache_write": 0.00, "output": 0.00}},   # EUR 0.12 / 1M
    "Clef-flash (local)":      {"provider": "clef_local", "model_id": "Cloudflare/clef-flash", "key_env": None, "reasoning": None,
                                "price": {"input": 0.0, "cached_input": 0.0, "cache_write": 0.0, "output": 0.0}},
}
INCLUDE_LOCAL = False   # True once the local loader is set up (revision pinned, loader read, quantization set); needs this machine's GPU
if not INCLUDE_LOCAL:
    MODELS.pop("Clef-flash (local)")
SYSTEM_ONE_PROVIDERS = {"typesafe", "cloudflare", "greenpt", "clef_local"}

@dataclass
class CallRecord:
    model: str
    convo_id: int
    variant: str                           # "original" (test), "dev", "perturbed" (probe) or "repeat" (probe control)
    raw_answer: str = ""
    answers: str = ""                      # JSON: question -> parsed label (None = invalid)
    probabilities: str = ""                # JSON: yes/no question -> P(yes), System One models only
    invalid_answers: int = 0
    latency_s: float = 0.0
    uncached_input_tokens: int = 0
    cached_input_tokens: int = 0
    cache_write_tokens: int = 0
    output_tokens: int = 0                 # includes reasoning tokens (billed as output)
    reasoning_tokens: int = 0
    cost_usd: float = 0.0
    served_model: str = ""
    error: str = ""
    timestamp: str = ""
    config_fingerprint: str = ""           # hash of prompt, wording version, schema, System One questions and settings
    input_sha: str = ""                    # hash of the exact transcript text sent
    identity_version: str = ""             # which identity wording was asked (also inside the fingerprint)
    impact: str = ""                       # JSON: GreenPT's per-call impact report (inferenceTime, energy, emissions, version)
    gpu_energy_wh: float = 0.0             # local model: whole-GPU energy during the call, from the driver's counter

def sha(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

# Evidence arm (section 13b): the LLMs also return the exact line their identity answer rests on
EVIDENCE_FIELD = "identity_evidence"
EVIDENCE_INSTRUCTIONS = (f"\n### {EVIDENCE_FIELD}\nCopy, word for word, the one line of the conversation that your identity_verified "
                         "answer rests on. Use an empty string if no line supports it.\n")

def llm_schema(evidence=False):
    properties = {name: {"type": "string", "enum": llm_option_keys(name)} for name in QUESTIONS}
    if evidence:
        properties[EVIDENCE_FIELD] = {"type": "string"}
    return {"type": "object", "properties": properties, "required": list(properties), "additionalProperties": False}

def use_identity_wording(version, evidence=False):
    # Swap the one changed question (and optionally add the evidence field), then rebuild everything rendered from
    # QUESTIONS, so prompts, schema, System One questions and the fingerprint all follow. Without the evidence field the
    # fingerprint payload is exactly what it was before the arm existed, so earlier rows still match.
    global LLM_QUESTION_BLOCK, LLM_SCHEMA, SYSTEM_ONE_QUESTIONS, PROMPT_FINGERPRINT, ACTIVE_IDENTITY_VERSION, EVIDENCE_ACTIVE
    QUESTIONS["identity_verified"] = {"type": "noul", **IDENTITY_WORDINGS[version]}
    LLM_QUESTION_BLOCK = f"{LLM_TASK}\n\n## Questions\n\n{render_question_block()}" + (EVIDENCE_INSTRUCTIONS if evidence else "")
    LLM_SCHEMA = llm_schema(evidence)
    SYSTEM_ONE_QUESTIONS = system_one_questions()
    payload = {"llm_question_block": LLM_QUESTION_BLOCK, "llm_conversation_template": LLM_CONVERSATION_TEMPLATE,
               "llm_schema": LLM_SCHEMA, "system_one_questions": SYSTEM_ONE_QUESTIONS, "identity_version": version}
    if evidence:
        payload["evidence_field"] = EVIDENCE_FIELD
    PROMPT_FINGERPRINT = sha(json.dumps(payload, sort_keys=True))
    ACTIVE_IDENTITY_VERSION, EVIDENCE_ACTIVE = version, evidence

use_identity_wording(FROZEN_IDENTITY_VERSION or "v1")

def config_fingerprint(model_name):
    # Prompt (with wording version), the model's own settings, the shared request settings and its provider's settings.
    # Prices and key names are excluded: cost is recomputed from logged tokens, and the key doesn't change the answer.
    spec = {key: value for key, value in MODELS[model_name].items() if key not in ("price", "key_env")}
    return sha(json.dumps({"prompt": PROMPT_FINGERPRINT, "model": spec, "request": REQUEST_SETTINGS,
                           "provider": PROVIDER_SETTINGS[spec["provider"]]}, sort_keys=True, default=str))

def api_key(env_name, required=True):
    # On Windows, read the saved user environment FIRST: keys rotate daily, and a process started before the
    # rotation still carries the old value in os.environ (every call then fails with a 401)
    key = None
    if os.name == "nt":
        import winreg
        try:
            with winreg.OpenKey(winreg.HKEY_CURRENT_USER, "Environment") as user_environment:
                key = winreg.QueryValueEx(user_environment, env_name)[0]
        except FileNotFoundError:
            key = None
    key = key or os.environ.get(env_name)
    assert key or not required, f"set {env_name}"
    return key

# Named like the SDKs' errors so the run loop stops on a rejected key instead of retrying it
class AuthenticationError(Exception):
    pass

class PermissionDeniedError(Exception):
    pass

def post_json(url, key, body):
    response = requests.post(url, headers={"Authorization": f"Bearer {key}"}, json=body, timeout=TRANSPORT["http_timeout_s"])
    if response.status_code == 401:
        raise AuthenticationError(f"401: {response.text[:300]}")
    if response.status_code in (402, 403):   # 402: no credits; 403: wrong region (Metis) or key without access
        raise PermissionDeniedError(f"{response.status_code}: {response.text[:300]}")
    response.raise_for_status()
    return response.json()

def call_claude(model_name, transcript, client):
    prefix, suffix = llm_prompt_parts(transcript)
    reasoning = MODELS[model_name]["reasoning"]
    settings = {"output_config": {"format": {"type": "json_schema", "schema": LLM_SCHEMA}}}
    if reasoning:
        settings["thinking"] = reasoning["thinking"]
        settings["output_config"]["effort"] = reasoning["effort"]
    response = client.messages.create(
        model=MODELS[model_name]["model_id"],
        max_tokens=REQUEST_SETTINGS["max_output_tokens"],
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prefix, "cache_control": PROVIDER_SETTINGS["anthropic"]["cache_control"]},
            {"type": "text", "text": suffix},
        ]}],
        **settings,
    )
    if response.stop_reason == "refusal":
        raise RuntimeError(f"refusal: {response.stop_details}")
    usage = response.usage
    return {
        "raw_answer": "".join(block.text for block in response.content if block.type == "text"),
        "uncached_input_tokens": usage.input_tokens,
        "cached_input_tokens": usage.cache_read_input_tokens or 0,
        "cache_write_tokens": usage.cache_creation_input_tokens or 0,
        "output_tokens": usage.output_tokens,
        "served_model": response.model,
    }

def call_gpt(model_name, transcript, client):
    prefix, suffix = llm_prompt_parts(transcript)
    settings = PROVIDER_SETTINGS["openai"]
    reasoning = {"reasoning": {"effort": MODELS[model_name]["reasoning"]}} if MODELS[model_name]["reasoning"] else {}
    response = client.responses.create(
        model=MODELS[model_name]["model_id"],
        input=[{"role": "user", "content": [
            {"type": "input_text", "text": prefix, "prompt_cache_breakpoint": settings["cache_breakpoint"]},
            {"type": "input_text", "text": suffix},
        ]}],
        **reasoning,
        text={"format": {"type": "json_schema", "name": "conversation_labels", "schema": LLM_SCHEMA, "strict": True}},
        max_output_tokens=REQUEST_SETTINGS["max_output_tokens"],
        prompt_cache_key=settings["prompt_cache_key"],
        store=settings["store"],
    )
    usage = response.usage
    cached = usage.input_tokens_details.cached_tokens or 0
    cache_written = getattr(usage.input_tokens_details, "cache_write_tokens", 0) or 0
    return {
        "raw_answer": response.output_text,
        "uncached_input_tokens": usage.input_tokens - cached - cache_written,
        "cached_input_tokens": cached,
        "cache_write_tokens": cache_written,
        "output_tokens": usage.output_tokens,
        "reasoning_tokens": usage.output_tokens_details.reasoning_tokens or 0,
        "served_model": response.model,
    }

def call_gemini(model_name, transcript, client):
    prefix, suffix = llm_prompt_parts(transcript)
    thinking_level = MODELS[model_name]["reasoning"]
    thinking = {"thinking_config": genai_types.ThinkingConfig(thinking_level=thinking_level)} if thinking_level else {}
    response = client.models.generate_content(
        model=MODELS[model_name]["model_id"],
        contents=prefix + PROVIDER_SETTINGS["google"]["joiner"] + suffix,
        config=genai_types.GenerateContentConfig(
            **thinking,
            response_mime_type="application/json",
            response_json_schema=LLM_SCHEMA,
            max_output_tokens=REQUEST_SETTINGS["max_output_tokens"],
        ),
    )
    usage = response.usage_metadata
    cached = usage.cached_content_token_count or 0
    thoughts = usage.thoughts_token_count or 0
    return {
        "raw_answer": response.text or "",
        "uncached_input_tokens": usage.prompt_token_count - cached,
        "cached_input_tokens": cached,
        "output_tokens": (usage.candidates_token_count or 0) + thoughts,
        "reasoning_tokens": thoughts,
        "served_model": response.model_version or "",
    }

def call_jev(model_name, transcript, client):
    response = client.system_one(state=transcript, questions=SYSTEM_ONE_QUESTIONS, model=MODELS[model_name]["model_id"])
    answers = {name: answer.model_dump() for name, answer in response.answers.items()}
    return {
        "raw_answer": json.dumps(answers),
        "system_one_answers": answers,
        "uncached_input_tokens": response.usage.input_tokens or 0,
        "output_tokens": response.usage.output_tokens or 0,
        "served_model": response.model,
    }

def call_cloudflare(model_name, transcript, client):
    url = PROVIDER_SETTINGS["cloudflare"]["url"].format(account=client["account"], model=MODELS[model_name]["model_id"])
    payload = post_json(url, client["key"], {"state": transcript, "questions": SYSTEM_ONE_QUESTIONS})
    body = payload.get("result", payload)   # Workers AI wraps responses in {"result": ..., "success": ..., "errors": ...}
    usage = body.get("usage") or {}
    return {
        "raw_answer": json.dumps(body["answers"]),
        "system_one_answers": body["answers"],
        "uncached_input_tokens": usage.get("input_tokens") or 0,   # 0 = not reported: section 12 flags it, cost is then unknown
        "served_model": body.get("model") or MODELS[model_name]["model_id"],
    }

def call_metis(model_name, transcript, client):
    payload = post_json(PROVIDER_SETTINGS["greenpt"]["url"], client,
                        {"model": MODELS[model_name]["model_id"], "state": transcript, "questions": SYSTEM_ONE_QUESTIONS})
    usage = payload.get("usage") or {}
    return {
        "raw_answer": json.dumps(payload["answers"]),
        "system_one_answers": payload["answers"],
        "uncached_input_tokens": usage.get("input_tokens") or 0,
        "output_tokens": usage.get("output_tokens") or 0,   # always 0 per GreenPT's docs
        "served_model": payload.get("model", ""),
        "impact": json.dumps(payload.get("impact") or {}),
    }

def gpu_energy_meter():
    # Whole-GPU energy counter from the driver (millijoules since it loaded); returns a reader in watt-hours
    import pynvml
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    return lambda: pynvml.nvmlDeviceGetTotalEnergyConsumption(handle) / 3.6e6

_clef_local = None
def load_clef_local(model_name):
    # The model card's release loader. joint_schema_model.py ships with the download: read it at the pinned revision first.
    global _clef_local
    if _clef_local is None:
        settings = PROVIDER_SETTINGS["clef_local"]
        assert settings["revision"], "pin PROVIDER_SETTINGS['clef_local']['revision'] to a commit hash, and read joint_schema_model.py at it"
        import torch
        from huggingface_hub import snapshot_download
        print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB (BF16 weights alone need ~18 GB)")
        path = snapshot_download(MODELS[model_name]["model_id"], revision=settings["revision"])
        sys.path.insert(0, path)
        import joint_schema_model
        loaded = joint_schema_model.load_release_model(path, device=settings["device"], **settings["load_kwargs"])
        model, processor = loaded if isinstance(loaded, tuple) else (loaded, None)   # return shape: confirm in the file
        _clef_local = {"model": model, "processor": processor, "systemone": joint_schema_model.systemone, "energy_wh": gpu_energy_meter()}
        _clef_local["systemone"](model, processor, {"state": "warm-up", "questions": SYSTEM_ONE_QUESTIONS})   # keep kernel compilation out of the timing
    return _clef_local

def call_clef_local(model_name, transcript, client):
    energy_before = client["energy_wh"]()
    response = client["systemone"](client["model"], client["processor"], {"state": transcript, "questions": SYSTEM_ONE_QUESTIONS})
    energy = client["energy_wh"]() - energy_before
    return {
        "raw_answer": json.dumps(response["answers"], default=float),
        "system_one_answers": response["answers"],
        "uncached_input_tokens": (response.get("usage") or {}).get("input_tokens", 0),
        "served_model": f"{MODELS[model_name]['model_id']}@{PROVIDER_SETTINGS['clef_local']['revision'][:12]}",
        "gpu_energy_wh": energy,
    }

PROVIDER_CALLERS = {"anthropic": call_claude, "openai": call_gpt, "google": call_gemini, "typesafe": call_jev,
                    "cloudflare": call_cloudflare, "greenpt": call_metis, "clef_local": call_clef_local}
MODEL_CALLERS = {model_name: PROVIDER_CALLERS[spec["provider"]] for model_name, spec in MODELS.items()}

def make_client(model_name):
    provider = MODELS[model_name]["provider"]
    if provider == "clef_local":
        return load_clef_local(model_name)   # loaded BEFORE timing starts
    key = api_key(MODELS[model_name]["key_env"])
    if provider == "anthropic":
        workspace_id = api_key("CLAUDE_WORKSPACE_ID", required=False)
        return anthropic.Anthropic(api_key=key, default_headers={"anthropic-workspace-id": workspace_id} if workspace_id else None)
    if provider == "cloudflare":
        return {"key": key, "account": api_key("CF_ACCOUNT_ID")}
    if provider == "greenpt":
        return key
    return {"openai": lambda: openai.OpenAI(api_key=key),
            "google": lambda: genai.Client(api_key=key),
            "typesafe": lambda: TypeSafeClient(api_key=key, model=MODELS[model_name]["model_id"], timeout=TRANSPORT["typesafe_timeout_s"])}[provider]()

def cost_usd(model_name, usage):
    price = MODELS[model_name]["price"]
    return (usage.get("uncached_input_tokens", 0) * price["input"] + usage.get("cached_input_tokens", 0) * price["cached_input"]
            + usage.get("cache_write_tokens", 0) * price["cache_write"] + usage.get("output_tokens", 0) * price["output"]) / 1_000_000

def label_one(model_name, convo_id, transcript, client, variant="original"):
    # One conversation per call. Latency = wall clock around the API call only.
    record = CallRecord(model=model_name, convo_id=int(convo_id), variant=variant, timestamp=datetime.now(timezone.utc).isoformat(),
                        config_fingerprint=config_fingerprint(model_name), input_sha=sha(transcript),
                        identity_version=ACTIVE_IDENTITY_VERSION)
    started = time.perf_counter()
    try:
        result = MODEL_CALLERS[model_name](model_name, transcript, client)
    except Exception as error:   # logged, never silently dropped; the run loop retries failures
        record.latency_s = time.perf_counter() - started
        record.error = f"{type(error).__name__}: {error}"[:500]
        return record
    record.latency_s = time.perf_counter() - started
    system_one_answers = result.pop("system_one_answers", None)
    for field_name, value in result.items():
        setattr(record, field_name, value)
    record.cost_usd = cost_usd(model_name, result)   # billed even if the answer turns out malformed
    try:   # a malformed response is logged as a failed call, never allowed to abort the run
        if system_one_answers is not None:
            labels, probabilities = parse_system_one_answers(system_one_answers)
            record.probabilities = json.dumps(probabilities)
        else:
            labels = parse_llm_answers(record.raw_answer)
    except Exception as error:
        record.error = f"MalformedResponse: {type(error).__name__}: {error}"[:500]
        return record
    record.answers = json.dumps(labels, default=lambda value: bool(value) if isinstance(value, np.bool_) else int(value))
    record.invalid_answers = sum(value is None for value in labels.values())
    return record

print({model_name: config_fingerprint(model_name) for model_name in MODELS})

### 7b. Harness export

Every identity wording, with and without the evidence field, rendered exactly as sent and written to `harness/` as one JSON file per version: the question block, conversation template, schema, System One questions, request and provider settings, model settings and each model's configuration fingerprint. Committed with the notebook, so a score change between runs can be traced to the model and not to a changed harness (a reader's point: most "nerfs" are harness changes). Nothing else is sent: no temperature, no top-p and no system prompt for any model (provider defaults; the instructions are in the user message).

In [ ]:
HARNESS_DIR = Path("harness")

def export_harness():
    HARNESS_DIR.mkdir(exist_ok=True)
    for version in IDENTITY_WORDINGS:
        for evidence in (False, True):
            use_identity_wording(version, evidence)
            record = {"identity_version": version, "evidence_field": evidence, "prompt_fingerprint": PROMPT_FINGERPRINT,
                      "config_fingerprints": {model_name: config_fingerprint(model_name) for model_name in MODELS},
                      "llm_question_block": LLM_QUESTION_BLOCK, "llm_conversation_template": LLM_CONVERSATION_TEMPLATE,
                      "llm_schema": LLM_SCHEMA, "system_one_questions": SYSTEM_ONE_QUESTIONS,
                      "request_settings": REQUEST_SETTINGS, "provider_settings": PROVIDER_SETTINGS,
                      "models": {model_name: {key: value for key, value in spec.items() if key != "key_env"} for model_name, spec in MODELS.items()},
                      "not_sent": "temperature, top_p and a system prompt are never sent (provider defaults)"}
            name = f"round2_identity_{version}{'_evidence' if evidence else ''}.json"
            (HARNESS_DIR / name).write_text(json.dumps(record, indent=2, sort_keys=True, default=str), encoding="utf-8")
    use_identity_wording(FROZEN_IDENTITY_VERSION or "v1")
    print(f"wrote {2 * len(IDENTITY_WORDINGS)} harness files to {HARNESS_DIR}/")

export_harness()

## 8. Smoke test

One call per model on one **dev** conversation, to prove the wiring before spending anything. For the new hosts it also settles what the docs leave open: the response shape of Workers AI, whether Metis answers from here (EU-only), whether each host reports token usage (no usage means no cost), and the local loader's signature. Read the raw answers, not just the parsed labels.

In [ ]:
RUN_SMOKE_TEST = False
SMOKE_MODELS = list(MODELS)   # e.g. ["Metis (GreenPT)"] to check one host
if RUN_SMOKE_TEST:
    smoke_row = dev.iloc[0]
    smoke_records = [label_one(model_name, smoke_row["convo_id"], smoke_row["text"], make_client(model_name)) for model_name in SMOKE_MODELS]
    smoke_table = pd.DataFrame([asdict(record) for record in smoke_records]).set_index("model")
    display(smoke_table[["served_model", "answers", "invalid_answers", "latency_s", "uncached_input_tokens",
                         "cached_input_tokens", "cache_write_tokens", "output_tokens", "cost_usd", "impact", "gpu_energy_wh", "error"]])
    for record in smoke_records:
        print(f"\n{record.model} raw answer:\n{record.raw_answer[:1200]}")
    print("truth:", {name: smoke_row[name] for name in QUESTIONS})

## 9. Run machinery

Same as round one: one conversation per call, sequential, same order for every model, every attempt appended to the log the moment it returns, resumable, and only rows whose fingerprint and input hash match the current notebook are ever reused or scored. Round two writes its own log, `call_log_round2.csv.gz`; round one's `call_log.csv` is never touched.

The log is gzip-compressed (90 MB as plain CSV after the full run, 7 MB compressed; GitHub refuses files over 100 MB). Each call is appended as its own small gzip member, which any gzip reader, pandas included, reads as one file; `compact_call_log()` rewrites it as a single member after a long run, so the appends don't erode the compression. A log path that doesn't end in `.gz` is read and appended as plain CSV, so per-run part files stay simple.

New here: per-host pacing (`TRANSPORT["min_interval_s"]`), because GreenPT allows 600 requests per 5 minutes and Metis can answer faster than that.

In [ ]:
import gzip
import shutil

CALL_LOG_PATH = Path("call_log_round2.csv.gz")
RUN_MODELS = list(MODELS)   # e.g. ["Jev 1.14"] to run one model at a time
MAX_ATTEMPTS = 3
LOG_COLUMNS = [field.name for field in fields(CallRecord)]
NUMERIC_COLUMNS = ["convo_id", "invalid_answers", "latency_s", "uncached_input_tokens", "cached_input_tokens",
                   "cache_write_tokens", "output_tokens", "reasoning_tokens", "cost_usd", "gpu_energy_wh"]

def open_log(mode):
    # gzip for the committed .csv.gz log (each append becomes a gzip member), plain text for anything else
    if CALL_LOG_PATH.suffix == ".gz":
        return gzip.open(CALL_LOG_PATH, mode + "t", newline="", encoding="utf-8")
    return CALL_LOG_PATH.open(mode, newline="", encoding="utf-8")

def compact_call_log():
    # Rewrite the gzip log as one member (best compression), via a temporary file so a crash can't lose it
    if CALL_LOG_PATH.suffix != ".gz" or not CALL_LOG_PATH.exists():
        return
    temporary = CALL_LOG_PATH.with_name(CALL_LOG_PATH.name + ".tmp")
    with gzip.open(CALL_LOG_PATH, "rb") as source, gzip.open(temporary, "wb", compresslevel=9) as target:
        shutil.copyfileobj(source, target)
    temporary.replace(CALL_LOG_PATH)

def append_to_log(record):
    is_new_log = not CALL_LOG_PATH.exists()
    with open_log("a") as log_file:
        writer = csv.DictWriter(log_file, fieldnames=LOG_COLUMNS)
        if is_new_log:
            writer.writeheader()
        writer.writerow(asdict(record))

def load_call_log():
    if not CALL_LOG_PATH.exists():
        return pd.DataFrame(columns=LOG_COLUMNS)
    call_log = pd.read_csv(CALL_LOG_PATH, dtype=str, keep_default_na=False)
    for column in NUMERIC_COLUMNS:
        call_log[column] = pd.to_numeric(call_log[column], errors="coerce")
    return call_log

def matching_calls(call_log, model_name, variant, frame, text_for, successful_only=True):
    # Rows for this model and variant, produced by the CURRENT configuration on the CURRENT input text
    # (successful ones only, unless every attempt is wanted, e.g. to count what failures cost)
    expected_input = {row["convo_id"]: sha(text_for(row)) for _, row in frame.iterrows()}
    rows = call_log[(call_log["model"] == model_name) & (call_log["variant"] == variant)
                    & (call_log["config_fingerprint"] == config_fingerprint(model_name))]
    if successful_only:
        rows = rows[rows["error"] == ""]
    return rows[rows["input_sha"] == rows["convo_id"].map(expected_input)]

def run(model_names, frame, variant="original", text_for=lambda row: row["text"]):
    for model_name in model_names:
        done = set(matching_calls(load_call_log(), model_name, variant, frame, text_for)["convo_id"])
        remaining = [row for _, row in frame.iterrows() if row["convo_id"] not in done]
        print(f"{model_name} [{variant}, identity {ACTIVE_IDENTITY_VERSION}]: {len(frame) - len(remaining)} already done, {len(remaining)} to run")
        if not remaining:
            continue
        client = make_client(model_name)
        pace = TRANSPORT["min_interval_s"].get(MODELS[model_name]["provider"], 0)
        for attempt in range(1, MAX_ATTEMPTS + 1):
            failed, last_error = [], ""
            for position, row in enumerate(remaining, start=1):
                started = time.perf_counter()
                record = label_one(model_name, row["convo_id"], text_for(row), client, variant)
                append_to_log(record)   # every attempt is logged, including the one that stops the run
                if record.error.startswith(("AuthenticationError", "PermissionDeniedError")):
                    raise RuntimeError(f"{model_name}: key rejected, stopping instead of retrying: {record.error[:200]}")
                if record.error:
                    failed.append(row)
                    last_error = record.error
                if position % 100 == 0:
                    print(f"  {position}/{len(remaining)} calls, {len(failed)} failed so far")
                time.sleep(max(0.0, pace - (time.perf_counter() - started)))
            if not failed:
                break
            print(f"  attempt {attempt}: {len(failed)} failed, last error: {last_error[:160]}")
            remaining = failed
            time.sleep(15 * attempt)

def as_wrong(question, truth):
    # The answer an invalid response is scored as: always wrong, never dropped (section 3)
    spec = QUESTIONS[question]
    if spec["type"] == "noul":
        return not truth
    if spec["type"] == "score":
        return 0 if truth >= len(spec["criteria"]) / 2 else len(spec["criteria"]) - 1   # the farthest level
    return "<invalid>"

def collect(variant="original", frame=test, text_for=lambda row: row["text"]):
    call_log = load_call_log()
    collected = {}
    for model_name in MODELS:
        calls = matching_calls(call_log, model_name, variant, frame, text_for)
        calls = calls.drop_duplicates("convo_id", keep="last").set_index("convo_id")   # same config: the latest answer
        if not set(frame["convo_id"]) <= set(calls.index):
            print(f"{model_name} [{variant}]: {len(calls)}/{len(frame)} answered with the current configuration, not scored yet")
            continue
        calls = calls.loc[frame["convo_id"]]
        answers = [json.loads(text) for text in calls["answers"]]
        probabilities = [json.loads(text) if text else {} for text in calls["probabilities"]]
        predictions = {}
        for question, spec in QUESTIONS.items():
            values = [as_wrong(question, truth) if answer.get(question) is None else answer[question]
                      for answer, truth in zip(answers, frame[question])]
            predictions[question] = np.array(values, dtype=object if spec["type"] == "choice" else None)
        model_probabilities = {question: np.array([p[question] for p in probabilities])
                               for question in NOUL_QUESTIONS if probabilities and question in probabilities[0]}
        collected[model_name] = (predictions, model_probabilities, calls)
    return collected

## 10. Dev: choosing the identity wording

The wording is chosen on a fixed sample of **300 dev conversations** (seed 42), never on test. Every model answers all seven questions for each version, so the table also shows whether the rewrite moved the other six answers.

The procedure, recorded so it can't drift:
1. Run `v0` (round one's wording, the reference) and `v1`.
2. Read `v1`'s remaining errors (the list below the table). Only then, if needed, write `v2`, and after reading its errors `v3`. At most three rewrites.
   - What happened: `v1` ("three details checked together") cut false alarms from ~92 to ~4 per model but raised misses from ~6 to ~58, because the three values in `verify-identity` come from the agent's system, not the conversation. `v2` is a reader's wording (avisangle), as written. `v3` describes the store's own policy (`kb.json`) and the clues that separate verified chats on the **train** split. On train, the policy alone (knowing the true request type) reaches F2 0.853, and the best possible rule on request type alone 0.859: no wording can be expected to reach 1.
3. Freeze the version with the best **mean identity F2 across all models** (not one model's best), unless it hurts the other questions; then set `FROZEN_IDENTITY_VERSION` in section 2 and re-run from the top.
   - **A change to this rule, disclosed.** Made on 2026-10-20, after seeing dev results for 6 of the 8 API models and before any test call. The rule as first written picked `v0` over `v3` by 0.001 mean F2 (0.825 vs 0.824), a tie at 300 chats that the rule gave no way to break. Added: a mean F2 within **0.005** of the best counts as a tie, and a tie goes to the version with **fewer total identity errors** (false alarms + misses, mean per model). The change was made knowing which version it favours, which is why it is stated here and in the Caveats. The cell below applies the rule mechanically, over the models that answered every version, and prints the outcome.
   - **Outcome, all 8 API models (2026-10-20):** `v3` has the best mean F2 outright (0.830 vs `v0`'s 0.827, `v1` 0.567, `v2` 0.473), so the rule as first written picks it too; the tie-break doesn't decide anything. It's still disclosed, because it was added after seeing results. `v3` makes about 38 identity errors per model on the 300 dev chats (17.8 false alarms, 20.1 misses) against `v0`'s 98 (91.8 and 6.0). **Frozen: `v3`.** Round one's wording (`v0`) and the reader's wording (`v2`) also run on test as declared secondary arms (section 11), so the headline can be checked against both.

The same dev calls later supply the decision models' probability thresholds (section 17), so those are chosen on dev too.

In [ ]:
DEV_SAMPLE_SIZE = 300
dev_sample = dev.sample(n=DEV_SAMPLE_SIZE, random_state=SEED).sort_values("convo_id")
DEV_VERSIONS = ["v0", "v1", "v2", "v3"]   # v2, v3 written after reading v1's dev errors (2026-10-20); no rewrites left
OTHER_QUESTIONS = [question for question in QUESTIONS if question != "identity_verified"]
F2_TIE = 0.005   # added 2026-10-20, after seeing dev results (disclosed above): mean F2 this close counts as a tie

RUN_DEV = False
if RUN_DEV:
    for version in DEV_VERSIONS:
        use_identity_wording(version)
        run(RUN_MODELS, dev_sample, variant="dev")

def yes_no_summary(truth, predicted):
    truth, predicted = np.asarray(truth, bool), np.asarray(predicted, bool)
    tp = int(np.sum(truth & predicted))
    return {"F2": f2(truth, predicted), "false alarms": int(np.sum(~truth & predicted)), "misses": int(np.sum(truth & ~predicted)),
            "precision": tp / max(predicted.sum(), 1), "recall": tp / max(truth.sum(), 1)}

dev_rows, dev_wrong = [], {}
for version in DEV_VERSIONS:
    use_identity_wording(version)
    for model_name, (predictions, _, _) in collect("dev", dev_sample).items():
        summary = yes_no_summary(dev_sample["identity_verified"].values, predictions["identity_verified"])
        others = np.mean([headline_metric(question, dev_sample[question].values, predictions[question]) for question in OTHER_QUESTIONS])
        dev_rows.append({"version": version, "model": model_name, **summary, "other six, mean headline": others})
        wrong = predictions["identity_verified"].astype(bool) != dev_sample["identity_verified"].values
        dev_wrong.setdefault(version, Counter()).update(dev_sample["convo_id"].values[wrong].tolist())
use_identity_wording(FROZEN_IDENTITY_VERSION or "v1")

if dev_rows:
    dev_table = pd.DataFrame(dev_rows)
    display(dev_table.pivot(index="model", columns="version", values=["F2", "false alarms", "misses"]).round(3))
    display(dev_table.groupby("version")[["F2", "false alarms", "misses", "other six, mean headline"]].mean().round(3)
            .rename(columns=lambda column: f"mean {column}"))
    # The freeze rule, applied mechanically over the models that answered every version (the tie-break was added on
    # 2026-10-20, after seeing 6 of 8 models' dev results and before any test call; see the text above)
    every_version = dev_table.groupby("model")["version"].nunique()
    compared = dev_table[dev_table["model"].isin(every_version[every_version == len(DEV_VERSIONS)].index)]
    rule = compared.groupby("version")[["F2", "false alarms", "misses"]].mean()
    rule["total errors"] = rule["false alarms"] + rule["misses"]
    tied = rule[rule["F2"] >= rule["F2"].max() - F2_TIE]
    print(f"\nfreeze rule over {compared['model'].nunique()} models: best mean F2 {rule['F2'].max():.3f}; "
          f"within {F2_TIE}: {list(tied.index)}; fewest total errors among those: {tied['total errors'].idxmin()}")
    display(rule.round(3))
    latest = DEV_VERSIONS[-1]
    models_scored = dev_table[dev_table["version"] == latest]["model"].nunique()
    print(f"\n{latest}: dev conversations most models still get wrong (read these before writing the next version)")
    for convo_id, wrong_count in dev_wrong.get(latest, Counter()).most_common(8):
        row = dev_sample[dev_sample["convo_id"] == convo_id].iloc[0]
        actions = sorted({turn["targets"][2] for turn in conversation_by_id[convo_id]["delexed"] if turn["speaker"] == "action"})
        print(f"\n--- convo {convo_id}: wrong for {wrong_count}/{models_scored} models; truth {row['identity_verified']}; logged actions {actions}")
        print(row["text"][:700])
else:
    print("no dev calls yet: set RUN_DEV = True")

## 11. The test run

1,000 test conversations x every model with the **frozen** identity wording. The cell refuses to run until `FROZEN_IDENTITY_VERSION` is set, so the test split can't be scored with a wording still being tuned.

**Declared secondary arms** (fixed 2026-10-20, before any test call): the same 1,000 chats with `v0` (round one's wording, run again today as a same-day control, since the cloud models can change between runs and weeks have passed since round one) and `v2` (the reader's wording). They are reported next to the headline in section 13a and never replace it; everything else in this notebook is scored with the frozen wording.

In [ ]:
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    assert FROZEN_IDENTITY_VERSION, "freeze the identity wording on dev first (section 10), then set FROZEN_IDENTITY_VERSION in section 2"
    use_identity_wording(FROZEN_IDENTITY_VERSION)
    run(RUN_MODELS, test)

SECONDARY_WORDINGS = ["v0", "v2"]   # declared 2026-10-20, before any test call
RUN_SECONDARY_WORDINGS = False
if RUN_SECONDARY_WORDINGS:
    assert FROZEN_IDENTITY_VERSION, "freeze the headline wording first"
    for version in SECONDARY_WORDINGS:
        use_identity_wording(version)
        run(RUN_MODELS, test)
    use_identity_wording(FROZEN_IDENTITY_VERSION)

## 10. Results

Every model with an answer for all 1,000 test conversations, scored with the shared scorers. Invalid answers are scored as wrong (see section 3), and each model's share of valid answers is reported. In the efficiency table, cost and the failure count cover every logged attempt for the current configuration and inputs, failed ones included, so retries can't hide; latency is per successful call.

New columns: whether a host reported token usage at all (if not, its cost is unknown, not zero), energy for the two hosts that measure it (GreenPT's reported figure, converted from its `Wms` (watt-milliseconds) to Wh, and the local GPU's measured Wh), and a check that Metis's `impact.version` and served model never changed during the run.

In [ ]:
if not FROZEN_IDENTITY_VERSION:
    print("NOTE: identity wording not frozen yet; everything below is provisional (scored with v1)")
use_identity_wording(FROZEN_IDENTITY_VERSION or "v1")

TO_WATT_HOURS = {"Wh": 1.0, "kWh": 1000.0, "mWh": 1e-3, "Ws": 1 / 3600, "J": 1 / 3600, "Wms": 1 / 3.6e6}

def impact_values(calls, field, conversion=None):
    # GreenPT's impact report, e.g. {"energy": {"total": 47467, "unit": "Wms"}} (watt-milliseconds). With a conversion
    # table, totals are converted by their unit; an unknown unit becomes NaN rather than a wrong number.
    values = []
    for text in calls["impact"]:
        value = json.loads(text).get(field) if text else None
        if isinstance(value, dict):
            total, unit = value.get("total", value.get("value")), value.get("unit")
            value = total * conversion.get(unit, np.nan) if conversion and total is not None else total
        values.append(value)
    return pd.to_numeric(pd.Series(values), errors="coerce")

headline_names = {"choice": "accuracy", "noul": "F2", "score": "accuracy"}
efficiency = {}
for model_name, (predictions, model_probabilities, calls) in collect().items():
    all_predictions[model_name] = predictions
    if model_probabilities:
        all_probabilities[model_name] = model_probabilities
    calls = calls.assign(cost_usd=[cost_usd(model_name, row) for row in calls.to_dict("records")])   # current price table
    total_input = calls[["uncached_input_tokens", "cached_input_tokens", "cache_write_tokens"]].sum(axis=1)
    attempts = matching_calls(load_call_log(), model_name, "original", test, lambda row: row["text"], successful_only=False)
    attempt_cost = sum(cost_usd(model_name, row) for row in attempts.to_dict("records"))   # failed attempts are billed too
    efficiency[model_name] = {
        "valid answers": 1 - calls["invalid_answers"].sum() / (len(calls) * len(QUESTIONS)),
        "failed attempts": int((attempts["error"] != "").sum()),
        "median latency s": calls["latency_s"].median(),
        "p90 latency s": calls["latency_s"].quantile(0.9),
        "cost per 1,000 conversations $": attempt_cost / len(calls) * 1000,
        "input tokens / call": total_input.mean(),
        "cache hit share": calls["cached_input_tokens"].sum() / max(total_input.sum(), 1),
        "output tokens / call": calls["output_tokens"].mean(),
        "reasoning tokens / call": calls["reasoning_tokens"].mean(),
        "served model": ", ".join(sorted(calls["served_model"].unique())),
        "usage reported": (total_input > 0).mean(),
        "identity false yeses per 1,000": np.sum(predictions["identity_verified"].astype(bool) & ~test["identity_verified"].values)
                                          / len(test) * 1000,
    }
    provider = MODELS[model_name]["provider"]
    if provider == "clef_local":
        efficiency[model_name]["GPU Wh per 1,000"] = calls["gpu_energy_wh"].mean() * 1000
    if provider == "greenpt":
        efficiency[model_name]["reported Wh per 1,000 (GreenPT)"] = impact_values(calls, "energy", TO_WATT_HOURS).mean() * 1000
        versions = {json.loads(text).get("version") for text in calls["impact"] if text}
        if len(versions) > 1 or calls["served_model"].nunique() > 1:
            print(f"WARNING {model_name}: the model behind the ID changed during the run: impact versions {versions}, "
                  f"served models {sorted(calls['served_model'].unique())}. Re-run it before comparing.")
    if efficiency[model_name]["usage reported"] < 1 and MODELS[model_name]["price"]["input"] > 0:
        print(f"WARNING {model_name}: {1 - efficiency[model_name]['usage reported']:.0%} of calls reported no input tokens; its cost is understated")

scores = {model_name: score_model(model_name) for model_name in all_predictions if model_name != "Keywords"}
scores["Keywords"] = {question: score_question(question, test[question].values, prediction)
                      for question, prediction in all_predictions["Keywords"].items()}

def question_table(question):
    return pd.DataFrame({model_name: model_scores[question] for model_name, model_scores in scores.items()
                         if question in model_scores}).T

for question in QUESTIONS:
    print(f"\n=== {question} ({QUESTIONS[question]['type']}) ===")
    display(question_table(question))
efficiency_table = pd.DataFrame(efficiency).T
display(efficiency_table)

### Paired comparisons against Jev 1.14

As in round one, but the reference is the newest Jev: each model against Jev 1.14 on the same conversations, question by question, with a paired permutation test (exact when 16 or fewer answers differ, otherwise 200,000 random swaps) and a Bonferroni correction over every comparison in this table. Jev 1.13 is in the table, so this is also the 1.13 vs 1.14 version check.

The three families in this notebook (this table, the before/after table in section 13, the three-host table in section 14) are each corrected for their own number of comparisons.

In [ ]:
import itertools

REFERENCE_MODEL = "Jev 1.14"
COMPARED_MODELS = [model for model in scores if model in MODELS and model != REFERENCE_MODEL]
COMPARISONS = len(COMPARED_MODELS) * len(QUESTIONS)
PERMUTATION_DRAWS = 200_000
EXACT_UP_TO = 16   # enumerate every swap pattern when this few conversations differ

def headline_counts(question, truth, predicted):
    # Per-conversation contributions to the headline metric: correctness for accuracy, (TP, FP, FN) for F2
    if QUESTIONS[question]["type"] == "noul":
        truth, predicted = np.asarray(truth, bool), np.asarray(predicted, bool)
        return np.stack([truth & predicted, ~truth & predicted, truth & ~predicted], axis=1).astype(float)
    return (np.asarray(truth) == np.asarray(predicted)).astype(float)[:, None]

def metric_from_counts(question, totals):
    if QUESTIONS[question]["type"] == "noul":
        return 5 * totals[..., 0] / np.maximum(5 * totals[..., 0] + 4 * totals[..., 2] + totals[..., 1], 1)
    return totals[..., 0] / len(test)

def paired_permutation(question, a, b, rng):
    # Two-sided: how often swapping each conversation's two answers at random opens a gap at least as large as observed.
    # Swapping conversation i moves (b_i - a_i) into A's totals and out of B's; only conversations that differ move anything.
    truth = test[question].values
    counts_a, counts_b = headline_counts(question, truth, a), headline_counts(question, truth, b)
    totals_a, totals_b = counts_a.sum(axis=0), counts_b.sum(axis=0)
    observed = metric_from_counts(question, totals_a) - metric_from_counts(question, totals_b)
    differing = np.flatnonzero(np.asarray(a) != np.asarray(b))
    if len(differing) == 0:
        return observed, 0, 1.0
    delta = counts_b[differing] - counts_a[differing]
    exact = len(differing) <= EXACT_UP_TO
    patterns = np.array(list(itertools.product([0.0, 1.0], repeat=len(differing)))) if exact else None
    total_draws = len(patterns) if exact else PERMUTATION_DRAWS
    hits = 0
    for start in range(0, total_draws, 20_000):
        size = min(20_000, total_draws - start)
        swaps = patterns[start:start + size] if exact else (rng.random((size, len(differing))) < 0.5).astype(float)
        moved = swaps @ delta
        gaps = metric_from_counts(question, totals_a + moved) - metric_from_counts(question, totals_b - moved)
        hits += int(np.sum(np.abs(gaps) >= abs(observed) - 1e-12))
    p = hits / total_draws if exact else (hits + 1) / (total_draws + 1)   # Monte Carlo: never report exactly 0
    return observed, len(differing), p

def paired_difference(question, model_a, model_b, rng, comparisons=None, a=None, b=None):
    # a / b: prediction arrays, when they aren't this round's all_predictions (round one's answers in section 13)
    a = all_predictions[model_a][question] if a is None else a
    b = all_predictions[model_b][question] if b is None else b
    difference, differing, p = paired_permutation(question, a, b, rng)
    adjusted = min(1.0, p * (comparisons or COMPARISONS))
    verdict = "clear after correction" if adjusted < 0.05 else "clear at 95% only" if p < 0.05 else "not distinguishable"
    return {"difference": difference * 100, "answers that differ": differing, "p": p, "Bonferroni p": adjusted, "verdict": verdict}

permutation_rng = np.random.default_rng(SEED)
comparison_rows = []
for model_name in (COMPARED_MODELS if REFERENCE_MODEL in scores else []):
    for question in QUESTIONS:
        comparison_rows.append({"model": model_name, "question": question, **paired_difference(question, model_name, REFERENCE_MODEL, permutation_rng)})
paired_vs_reference = pd.DataFrame(comparison_rows)
if len(paired_vs_reference):
    display(paired_vs_reference.pivot(index="question", columns="model", values="difference").round(1)
            .astype(str) + " (" + paired_vs_reference.pivot(index="question", columns="model", values="verdict") + ")")
paired_vs_reference

## 13. Before and after: did one definition fix it?

Round one's scored test answers (`results/test_predictions.csv`, exported by `02_benchmark.ipynb` from its fingerprint-matched rows) against round two's, for the four models that ran in both: Claude Haiku 5.5, GPT-6 Luna, Gemini 3.5 Flash-Lite and Jev 1.13 (round one's "Jev", the same pinned `jev-1.13.0`). Same 1,000 conversations, same truth, so every comparison is paired.

- **The fix:** identity F2, false alarms and misses, round one vs round two.
- **The collateral check:** all seven questions are answered in one call, so a new definition could move the other six. For each, the number of answers that changed and a paired test of the score change. A significant change on an untouched question is reported, not explained away. Note that the cloud LLMs aren't deterministic: some answers change on a plain re-run (the repeat control in section 15 measures how many), so "answers that changed" is an upper bound on what the wording caused.

Bonferroni over 4 models x 7 questions = 28 comparisons.

In [ ]:
ROUND_ONE_PREDICTIONS = Path("results") / "test_predictions.csv"
ROUND_ONE_NAMES = {"Claude Haiku 5.5": "Claude Haiku 5.5", "GPT-6 Luna": "GPT-6 Luna",
                   "Gemini 3.5 Flash-Lite": "Gemini 3.5 Flash-Lite", "Jev 1.13": "Jev"}   # round two name -> round one name
round_one_export = pd.read_csv(ROUND_ONE_PREDICTIONS, dtype=str)

def round_one_predictions(round_one_name):
    rows = round_one_export[round_one_export["model"] == round_one_name]
    predictions = {}
    for question, spec in QUESTIONS.items():
        by_convo = rows[rows["question"] == question].set_index(rows.loc[rows["question"] == question, "convo_id"].astype(int))
        assert set(test["convo_id"]) <= set(by_convo.index), f"{round_one_name} {question}: round one export incomplete"
        truth = by_convo.loc[test["convo_id"], "truth"]
        values = by_convo.loc[test["convo_id"], "prediction"]
        if spec["type"] == "noul":
            assert ((truth == "True").values == test[question].values).all(), f"{question}: round one truth differs"
            predictions[question] = (values == "True").values
        elif spec["type"] == "score":
            predictions[question] = values.astype(int).values
        else:
            predictions[question] = values.values.astype(object)
    return predictions

paired_models = [model for model in ROUND_ONE_NAMES if model in all_predictions]
round_one = {model: round_one_predictions(ROUND_ONE_NAMES[model]) for model in paired_models}
BEFORE_AFTER_COMPARISONS = len(ROUND_ONE_NAMES) * len(QUESTIONS)

identity_rows, collateral_rows = [], []
before_after_rng = np.random.default_rng(SEED)
for model_name in paired_models:
    truth = test["identity_verified"].values
    before = yes_no_summary(truth, round_one[model_name]["identity_verified"])
    after = yes_no_summary(truth, all_predictions[model_name]["identity_verified"])
    identity_rows.append({"model": model_name, **{f"{key}, round one": value for key, value in before.items()},
                          **{f"{key}, round two": value for key, value in after.items()}})
    for question in QUESTIONS:
        result = paired_difference(question, model_name, model_name, before_after_rng, comparisons=BEFORE_AFTER_COMPARISONS,
                                   a=all_predictions[model_name][question], b=round_one[model_name][question])
        collateral_rows.append({"model": model_name, "question": question, **result})

if identity_rows:
    identity_before_after = pd.DataFrame(identity_rows).set_index("model")
    display(identity_before_after[[f"{key}, round {r}" for key in ["F2", "false alarms", "misses", "precision", "recall"] for r in ["one", "two"]]].round(3))
    before_after = pd.DataFrame(collateral_rows)
    print("Round two minus round one (points of accuracy, or F2 x 100), and the paired verdict")
    display(before_after.pivot(index="question", columns="model", values="difference").round(1).astype(str)
            + " (" + before_after.pivot(index="question", columns="model", values="answers that differ").astype(str) + " differ, "
            + before_after.pivot(index="question", columns="model", values="verdict") + ")")
else:
    print("no round-two test answers for the paired models yet")

## 13a. The identity question on test, by wording

The headline wording next to the two declared secondary arms (section 11) and round one's answers, per model, on the same 1,000 test conversations. F2 is a 0-1 score for yes/no labels where a miss costs 4x a false alarm; false alarms (said verified, it wasn't) and misses (verified, said no) are the counts behind it. `v0` today vs round one isolates run-to-run and over-time change, since the wording is identical.

In [ ]:
HEADLINE = FROZEN_IDENTITY_VERSION or "v1"
wording_rows = []
for version in [HEADLINE] + [v for v in SECONDARY_WORDINGS if v != HEADLINE]:
    use_identity_wording(version)
    label = f"{version} (headline)" if version == HEADLINE else version
    for model_name, (predictions, _, _) in collect("original", test).items():
        wording_rows.append({"model": model_name, "wording": label,
                             **yes_no_summary(test["identity_verified"].values, predictions["identity_verified"])})
use_identity_wording(HEADLINE)
for model_name in paired_models:
    wording_rows.append({"model": model_name, "wording": "v0, round one's calls",
                         **yes_no_summary(test["identity_verified"].values, round_one[model_name]["identity_verified"])})
if wording_rows:
    by_wording = pd.DataFrame(wording_rows)
    display(by_wording.pivot(index="model", columns="wording", values=["F2", "false alarms", "misses"]).round(3))
else:
    print("no test answers yet")

## 13b. The evidence arm: make the model show its line

A reader's suggestion (xapep): make the model output the exact line its "verified" answer rests on, then check that line against the transcript before trusting the answer. The cheap LLMs run the 1,000 test chats once more with the frozen wording plus one extra output field, `identity_evidence`. A "yes" whose quote isn't found in the transcript (compared ignoring case and spacing, a leading speaker name and surrounding quote marks; at least 8 characters) becomes "no". Other answers are left alone.

Three columns per model, so each effect is separate: **no field** (the plain test run), **with field** (the answers as given, which shows whether asking for a quote changes the answer by itself), and **after the check**. Plus how many false yeses the check removed and how many true yeses it cost. Decision models can't quote text, so they are the comparison: a different route to the same reliability.

A prediction to test, not assume: round one's false yeses came from real lines ("May I have your full name?"), so a verbatim check alone may not remove many of them.

In [ ]:
EVIDENCE_MODELS = [model_name for model_name, spec in MODELS.items() if spec["provider"] in ("anthropic", "openai", "google")]
RUN_EVIDENCE_ARM = False
if RUN_EVIDENCE_ARM:
    assert FROZEN_IDENTITY_VERSION, "freeze the identity wording on dev first (section 10)"
    use_identity_wording(FROZEN_IDENTITY_VERSION, evidence=True)
    run(EVIDENCE_MODELS, test, variant="evidence")
use_identity_wording(FROZEN_IDENTITY_VERSION or "v1", evidence=True)
evidence_collected = {model_name: result for model_name, result in collect("evidence", test).items() if model_name in EVIDENCE_MODELS}
use_identity_wording(FROZEN_IDENTITY_VERSION or "v1")

QUOTE_MARKS = "\"'\u201c\u201d\u2018\u2019 "

def normalized(text):
    return re.sub(r"\s+", " ", text.lower()).strip()

def quote_found(quote, transcript):
    quote = re.sub(r"^(agent|customer)\s*:\s*", "", normalized(quote).strip(QUOTE_MARKS))
    return len(quote) >= 8 and quote in normalized(transcript)

def evidence_quote(raw_answer):
    try:
        answer = json.loads(raw_answer)
    except (json.JSONDecodeError, TypeError):
        return ""
    return (answer.get(EVIDENCE_FIELD) or "") if isinstance(answer, dict) else ""

evidence_rows, surviving_examples = [], {}
identity_truth = test["identity_verified"].values.astype(bool)
for model_name, (predictions, _, calls) in evidence_collected.items():
    quotes = [evidence_quote(raw) for raw in calls["raw_answer"]]
    found = np.array([quote_found(quote, text) for quote, text in zip(quotes, test["text"])])
    with_field = predictions["identity_verified"].astype(bool)
    checked = with_field & found
    row = {"model": model_name}
    if model_name in all_predictions:
        plain = all_predictions[model_name]["identity_verified"].astype(bool)
        row.update({"F2, no field": f2(identity_truth, plain), "false yeses, no field": int(np.sum(plain & ~identity_truth)),
                    "misses, no field": int(np.sum(~plain & identity_truth))})
    row.update({"F2, with field": f2(identity_truth, with_field), "false yeses, with field": int(np.sum(with_field & ~identity_truth)),
                "misses, with field": int(np.sum(~with_field & identity_truth)),
                "F2, after check": f2(identity_truth, checked), "false yeses, after check": int(np.sum(checked & ~identity_truth)),
                "misses, after check": int(np.sum(~checked & identity_truth)),
                "yes answers with no matching quote": int(np.sum(with_field & ~found)),
                "false yeses removed by the check": int(np.sum(with_field & ~found & ~identity_truth)),
                "true yeses lost to the check": int(np.sum(with_field & ~found & identity_truth))})
    evidence_rows.append(row)
    surviving_examples[model_name] = [quote for quote, kept, real in zip(quotes, checked, identity_truth) if kept and not real][:3]

if evidence_rows:
    evidence_table = pd.DataFrame(evidence_rows).set_index("model")
    print("F2: 0-1 score for yes/no labels where a miss costs 4x a false yes")
    display(evidence_table.round(3).T)
    for model_name, examples in surviving_examples.items():
        print(f"\n{model_name}: quotes behind false yeses that PASSED the check (first 3)")
        for quote in examples:
            print(f"  - {quote[:160]}")
else:
    print("no evidence-arm answers yet: set RUN_EVIDENCE_ARM = True after freezing the wording")

## 14. Same weights, three hosts

Clef-flash is open weights, so the same 9B model is served by Cloudflare, by GreenPT as Metis (full precision, per GreenPT) and by the local GPU (quantized to fit 16 GB). If hosting didn't matter, all three would give the same answers to the same 1,000 conversations.

For each pair of hosts and each question: how often the labels agree, how far apart the yes/no probabilities are (mean and largest absolute difference in P(yes)), and the paired test of the score difference (Bonferroni over 3 pairs x 7 questions = 21).

Cloudflare returns probabilities rounded to 4 decimals and GreenPT unrounded (smoke test, 2026-10-19), so tiny probability gaps between them are rounding, and a P(yes) within 0.00005 of 0.5 could flip a label on rounding alone. Read the local results as **quantization**, not hosting: the two hosted copies can be compared as hosts; the local copy differs in precision too. Cloudflare doesn't state its serving precision; close agreement with Metis would suggest full precision there too, but can't prove it.

In [ ]:
HOSTS = ["Clef-flash (Cloudflare)", "Metis (GreenPT)", "Clef-flash (local)"]
hosts_scored = [host for host in HOSTS if host in all_predictions]
host_pairs = list(itertools.combinations(hosts_scored, 2))
HOST_COMPARISONS = 3 * len(QUESTIONS)
host_rng = np.random.default_rng(SEED)
host_rows = []
for host_a, host_b in host_pairs:
    for question in QUESTIONS:
        a, b = all_predictions[host_a][question], all_predictions[host_b][question]
        row = {"pair": f"{host_a} vs {host_b}", "question": question, "label agreement": np.mean(a == b),
               **paired_difference(question, host_a, host_b, host_rng, comparisons=HOST_COMPARISONS)}
        if question in NOUL_QUESTIONS:
            gap = np.abs(all_probabilities[host_a][question] - all_probabilities[host_b][question])
            row.update({"mean |P(yes) gap|": gap.mean(), "largest |P(yes) gap|": gap.max()})
        host_rows.append(row)

if host_rows:
    host_table = pd.DataFrame(host_rows)
    display(host_table.pivot(index="question", columns="pair", values="label agreement").round(3))
    display(host_table)
    host_efficiency = pd.DataFrame(efficiency).T.reindex(hosts_scored)
    display(host_efficiency[[column for column in ["median latency s", "p90 latency s", "cost per 1,000 conversations $",
                                                   "reported Wh per 1,000 (GreenPT)", "GPU Wh per 1,000"]
                             if column in host_efficiency]])
else:
    print("fewer than two Clef-flash hosts scored yet")

## 15. Memorization probe and repeat control

ABCD has been public since 2021. If a model has memorized it, it could label a conversation by recognizing it rather than reading it. The probe swaps every identifying detail (names, usernames, emails, phone numbers, order and account IDs, addresses, zip codes) for fresh fake values on 200 test conversations and runs them again.

**What it can and can't show.** It tests robustness to these substitutions. It does not rule out memorization: the conversation's structure and most of its wording are untouched, and 31 of the 200 sampled conversations contain no identifying detail at all (mostly FAQ questions), so they go through unchanged. The analysis below uses only the conversations that actually changed, reports a paired interval for the change, and counts individual flipped labels, since a flat average can hide errors that cancel out. Without a repeat run on the unchanged text, some flips are ordinary run-to-run variation rather than an effect of the swap; the optional control below measures that.

The swap is an exact-string match against the scenario, so a customer's own typo of their name ("Rodriquez" for "rodriguez") slips through; the IDs, which carry the recognizable detail, are always swapped.

**Round two runs the repeat control** (`variant="repeat"`): the same 200 unchanged conversations a second time. Flips on a plain repeat are run-to-run noise; only flips beyond that level can be blamed on the swap. Round one couldn't make that split.

In [ ]:
PROBE_SIZE = 200
probe_rows = test.sample(n=PROBE_SIZE, random_state=SEED).sort_values("convo_id")
fake_rng = random.Random(SEED)
FAKE_FIRST = ["jordan", "riley", "casey", "morgan", "avery", "quinn", "rowan", "emerson", "sage", "harper"]
FAKE_LAST = ["okafor", "lindqvist", "moreau", "tanaka", "castillo", "novak", "brennan", "haddad", "ivanova", "mensah"]

def fake_like(value):
    # Same shape, new content: digits -> random digits, letters -> random letters
    return "".join(str(fake_rng.randint(0, 9)) if ch.isdigit() else fake_rng.choice("abcdefghijklmnopqrstuvwxyz") if ch.isalpha() else ch
                   for ch in value)

def perturb(convo_id):
    scenario = conversation_by_id[convo_id]["scenario"]
    replacements = {}
    first, *rest = scenario["personal"]["customer_name"].split()
    new_first, new_last = fake_rng.choice(FAKE_FIRST), fake_rng.choice(FAKE_LAST)
    replacements[scenario["personal"]["customer_name"]] = f"{new_first} {new_last}"
    replacements[first] = new_first
    for last in rest:
        replacements[last] = new_last
    for field in ["email", "phone", "username", "account_id"]:
        if scenario["personal"].get(field):
            replacements[scenario["personal"][field]] = fake_like(scenario["personal"][field])
    for field in ["order_id", "zip_code", "street_address", "full_address"]:
        if scenario["order"].get(field):
            replacements[scenario["order"][field]] = fake_like(scenario["order"][field])
    text = test.loc[test["convo_id"] == convo_id, "text"].iloc[0]
    for original in sorted(replacements, key=len, reverse=True):   # longest first, so full names go before first names
        text = re.sub(re.escape(original), replacements[original], text, flags=re.IGNORECASE)
    return text

perturbed_text = {convo_id: perturb(convo_id) for convo_id in probe_rows["convo_id"]}
example_id = next(convo_id for convo_id in probe_rows["convo_id"]   # show one where something was actually swapped
                  if perturbed_text[convo_id] != test.loc[test["convo_id"] == convo_id, "text"].iloc[0])
print(test.loc[test["convo_id"] == example_id, "text"].iloc[0][:600], "\n---\n", perturbed_text[example_id][:600])

perturbed_for = lambda row: perturbed_text[row["convo_id"]]
RUN_MEMORIZATION_PROBE = False   # needs the frozen wording, like the test run
if RUN_MEMORIZATION_PROBE:
    run(RUN_MODELS, probe_rows, variant="perturbed", text_for=perturbed_for)
# Control: the SAME unchanged text run a second time, to measure how many labels flip with no swap at all
RUN_REPEAT_CONTROL = False
if RUN_REPEAT_CONTROL:
    run(RUN_MODELS, probe_rows, variant="repeat")

changed_ids = [convo_id for convo_id in probe_rows["convo_id"]
               if perturbed_text[convo_id] != test.loc[test["convo_id"] == convo_id, "text"].iloc[0]]
print(f"{len(changed_ids)} of {len(probe_rows)} sampled conversations actually changed; the rest had no identifying detail")
changed_mask = probe_rows["convo_id"].isin(changed_ids).values
probe_positions = test.index[test["convo_id"].isin(probe_rows["convo_id"])]
probe_resamples = np.random.default_rng(SEED).integers(0, changed_mask.sum(), size=(BOOTSTRAP_RESAMPLES, changed_mask.sum()))

def per_conversation_accuracy(predictions, truth_frame):
    return np.mean([predictions[question] == truth_frame[question].values for question in QUESTIONS], axis=0)

def label_flips(first, second):
    return int(sum(np.sum(first[question] != second[question]) for question in QUESTIONS))

perturbed, repeated = collect("perturbed", probe_rows, perturbed_for), collect("repeat", probe_rows)
probe_results = {}
for model_name, (predictions, _, _) in perturbed.items():
    original = {question: all_predictions[model_name][question][probe_positions][changed_mask] for question in QUESTIONS}
    swapped = {question: predictions[question][changed_mask] for question in QUESTIONS}
    truth = probe_rows[changed_mask]
    change = per_conversation_accuracy(swapped, truth) - per_conversation_accuracy(original, truth)
    low, high = np.percentile(change[probe_resamples].mean(axis=1), [2.5, 97.5])
    probe_results[model_name] = {
        "mean accuracy, original": per_conversation_accuracy(original, truth).mean(),
        "mean accuracy, details swapped": per_conversation_accuracy(swapped, truth).mean(),
        "change": change.mean(), "95% paired interval": f"{low:+.3f} to {high:+.3f}",
        "labels flipped by the swap": f"{label_flips(original, swapped)} / {changed_mask.sum() * len(QUESTIONS)}",
    }
    if model_name in repeated:
        repeat = {question: repeated[model_name][0][question][changed_mask] for question in QUESTIONS}
        probe_results[model_name]["labels flipped on a plain repeat"] = f"{label_flips(original, repeat)} / {changed_mask.sum() * len(QUESTIONS)}"
pd.DataFrame(probe_results).T

## 16. Cost and energy at scale

The measured cost per conversation, projected to **10 million conversations a month**. Latency is the median single call measured here; parallel throughput is a separate experiment. Energy is shown only where it is measured: GreenPT's own per-call report for Metis, and the GPU's energy counter for the local model. They don't measure the same thing (a data-centre figure vs one consumer GPU), so they sit side by side, not on one scale.

In [ ]:
MONTHLY_CONVERSATIONS = 10_000_000
if len(efficiency_table):
    at_scale = efficiency_table[["cost per 1,000 conversations $", "median latency s"]].copy()
    at_scale["cost per month at 10M $"] = at_scale["cost per 1,000 conversations $"] * MONTHLY_CONVERSATIONS / 1000
    # A reader's point (xapep): a wrong "verified" can cost more than the labeling itself, so false yeses sit next to cost
    at_scale["identity false yeses per 1,000"] = efficiency_table["identity false yeses per 1,000"].astype(float)
    at_scale["identity false yeses per month at 10M"] = at_scale["identity false yeses per 1,000"] * MONTHLY_CONVERSATIONS / 1000
    for column in ["reported Wh per 1,000 (GreenPT)", "GPU Wh per 1,000"]:
        if column in efficiency_table:
            at_scale[column.replace("per 1,000", "per month at 10M")] = efficiency_table[column].astype(float) * MONTHLY_CONVERSATIONS / 1000
    display(at_scale.sort_values("cost per month at 10M $"))

## 17. Precision and recall: which mistake is expensive?

F2 is one number, but a business decides which error hurts. For every yes/no label there are two mistakes: a **false alarm** (saying yes when it was no, which hurts precision) and a **miss** (saying no when it was yes, which hurts recall).

| Label | A plausible business use | The expensive mistake | So watch |
|---|---|---|---|
| Identity verified? | Compliance audit: find calls where the agent skipped verification | Saying "verified" when it wasn't hides exactly the calls the audit is looking for | **precision** |
| Refund issued? | Reconcile refunds against the payments system | A missed refund leaves money unaccounted for; a false one sends finance chasing nothing | **recall**, then precision |
| Promo code given? | Catch promo-code abuse or leakage | A missed promo is lost money that never gets reviewed | **recall** |
| Handed to another team? | Make sure every escalation gets a follow-up | A missed handoff is a customer left waiting; a false one is a wasted check | **recall** |

The LLMs return a single yes or no, so each sits at one fixed point. The decision models (and the supervised model) return a probability, so the threshold can be moved to buy precision with recall or the other way round; the figures show that whole curve.

**Thresholds are chosen on dev, then applied to test once** (round one's audit finding 6). For each precision target, the threshold is the one with the highest dev recall whose dev precision reaches the target, using the 300 dev conversations asked with the frozen wording. The table reports what that fixed threshold scores on test: a deployment estimate, not round one's test-set envelope. A target the dev curve can't reach is left blank.

In [ ]:
from sklearn.metrics import precision_recall_curve

pr_rows = []
for question in NOUL_QUESTIONS:
    truth = test[question].values.astype(bool)
    for model_name, model_scores in scores.items():
        if question not in model_scores:
            continue
        predicted = np.asarray(all_predictions[model_name][question], bool)
        pr_rows.append({"question": question, "model": model_name,
                        "precision": model_scores[question]["precision"], "recall": model_scores[question]["recall"],
                        "false alarms": int(np.sum(predicted & ~truth)), "misses": int(np.sum(~predicted & truth))})
precision_recall_table = pd.DataFrame(pr_rows)
display(precision_recall_table.pivot(index="model", columns="question", values=["precision", "recall"]).round(2))

# Deployment estimate: threshold picked on dev (frozen wording), scored once on test
use_identity_wording(FROZEN_IDENTITY_VERSION or "v1")
dev_collected = collect("dev", dev_sample)
dev_probability_models = [model for model in all_probabilities if model in dev_collected]
dev_probability_models += [SUPERVISED]
supervised_dev_probabilities = {question: train_tfidf(train, question).predict_proba(dev_sample["text"])[:, 1] for question in NOUL_QUESTIONS}

def dev_threshold(model_name, question, target_precision):
    probabilities = supervised_dev_probabilities[question] if model_name == SUPERVISED else dev_collected[model_name][1][question]
    precision, recall, thresholds = precision_recall_curve(dev_sample[question].values.astype(bool), probabilities)
    reachable = np.flatnonzero(precision[:-1] >= target_precision)   # the last point has no threshold
    return thresholds[reachable[np.argmax(recall[reachable])]] if len(reachable) else None

threshold_rows = []
for question in NOUL_QUESTIONS:
    truth = test[question].values.astype(bool)
    for model_name in dev_probability_models:
        for target in [0.80, 0.90, 0.95]:
            threshold = dev_threshold(model_name, question, target)
            row = {"question": question, "model": model_name, "target precision": target, "dev threshold": threshold}
            if threshold is not None:
                predicted = all_probabilities[model_name][question] >= threshold
                tp = int(np.sum(truth & predicted))
                row.update({"test precision": tp / max(predicted.sum(), 1), "test recall": tp / max(truth.sum(), 1)})
            threshold_rows.append(row)
print("Thresholds chosen on the 300 dev conversations, scored once on test")
pd.DataFrame(threshold_rows).round(3)

### Export

Every scored round-two prediction (one row per model, conversation and question, with the truth) and every scored call's latency, tokens, cost and energy, from fingerprint-matched rows only. Round one's `results/test_*.csv` are left as they are: section 13 reads them.

In [ ]:
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
prediction_rows = []
for model_name, predictions in all_predictions.items():
    for question, values in predictions.items():
        for convo_id, truth, value in zip(test["convo_id"], test[question], values):
            prediction_rows.append({"model": model_name, "convo_id": convo_id, "question": question,
                                    "truth": truth, "prediction": value})
pd.DataFrame(prediction_rows).to_csv(RESULTS_DIR / "round2_test_predictions.csv", index=False)

call_columns = ["model", "convo_id", "latency_s", "uncached_input_tokens", "cached_input_tokens", "cache_write_tokens",
                "output_tokens", "reasoning_tokens", "cost_usd", "served_model", "impact", "gpu_energy_wh"]
scored_calls = pd.concat([calls.reset_index()[call_columns] for _, (_, _, calls) in collect().items()])
scored_calls.to_csv(RESULTS_DIR / "round2_test_calls.csv", index=False)
print(f"wrote {len(prediction_rows):,} predictions and {len(scored_calls):,} calls to {RESULTS_DIR}/")

## Figures

Round one's style, saved to `figures/round2/` (200 dpi, video-ready). Colour follows the model on every chart, in the reference palette's fixed slot order: Jev 1.13 blue (round one's Jev colour), Jev 1.14 orange, GPT-6 Luna aqua, Gemini Flash-Lite yellow, Claude Haiku 5.5 green, Clef red, and **one violet for all three Clef-flash hosts** (they are the same weights). The hosts are told apart by texture and name: Cloudflare solid, Metis hatched `//`, local cross-hatched; dashed and dotted lines on line charts. Baselines are grey. Values are printed on the marks, so no reading depends on colour alone.

- **03** every question, every model, with 95% intervals
- **04** cost per 1,000 conversations and **05** latency
- **07** the memorization probe
- **08** precision vs recall for every yes/no label, with the threshold curves of Jev 1.14, Clef, Clef-flash and the supervised model
- **09** latency: every call
- **10** call-reason routing
- **11** the identity fix: false alarms and misses, round one vs round two
- **12** same weights, three hosts: label agreement by question

In [ ]:
import math

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

FIGURES_DIR = Path("figures") / "round2"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
INK, MUTED_INK, AXIS, GRID, SURFACE = "#0b0b0b", "#52514e", "#c3c2b7", "#e5e4df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": MUTED_INK, "xtick.color": MUTED_INK, "ytick.color": MUTED_INK,
    "text.color": INK, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "legend.frameon": False,
})

# Fixed slot order from the validated reference palette (validator: all checks pass in this order)
CLEF_FLASH_VIOLET = "#6250d6"   # one hue for the three hosts of the same weights; texture tells them apart
MODEL_COLORS = {
    "Jev 1.13": "#2a78d6", "Jev 1.14": "#eb6834", "GPT-6 Luna": "#1baf7a", "Gemini 3.5 Flash-Lite": "#eda100",
    "Claude Haiku 5.5": "#008300", "Clef": "#e34948", "Clef-flash (Cloudflare)": CLEF_FLASH_VIOLET,
    "Metis (GreenPT)": CLEF_FLASH_VIOLET, "Clef-flash (local)": CLEF_FLASH_VIOLET,
    SUPERVISED: "#52514e", "Keywords": "#b5b4ae",
}
MODEL_HATCHES = {"Metis (GreenPT)": "///", "Clef-flash (local)": "xxx"}
MODEL_LINESTYLES = {"Metis (GreenPT)": "--", "Clef-flash (local)": ":"}
DISPLAY_NAMES = {SUPERVISED: f"TF-IDF + LR ({len(train):,} labels)"}

def texture(bars, models):
    # Hatching in the surface colour on the bar, so the shared violet still separates the three hosts
    for bar, model in zip(bars, models):
        if model in MODEL_HATCHES:
            bar.set_hatch(MODEL_HATCHES[model])
            bar.set_edgecolor(SURFACE)
            bar.set_linewidth(0)
CHART_MODELS = [model for model in MODEL_COLORS if model in scores]   # baselines last, fixed order
ZERO_SHOT_CHART_MODELS = [model for model in CHART_MODELS if model in MODELS]
QUESTION_TITLES = {
    "reason": "Reason for contact (10 options)", "intent": "Specific intent (55 options)",
    "resolution": "How far the agent went (3 levels)", "identity_verified": "Identity verified?",
    "refund_offered": "Refund issued?", "promo_given": "Promo code given?", "handed_off": "Handed to another team?",
}

def display_name(model_name):
    return DISPLAY_NAMES.get(model_name, model_name)

def style_value_axis(ax, axis="x"):
    ax.grid(axis=axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)

def save_figure(fig, name):
    path = FIGURES_DIR / f"{name}.png"
    for attempt in range(5):   # Windows: a file watcher can hold a just-written PNG for a moment (OSError 22)
        try:
            fig.savefig(path, dpi=200, bbox_inches="tight")
            break
        except OSError:
            if attempt == 4:
                raise
            time.sleep(0.5)
    plt.show()
    print(f"saved {path}")

# Column charts, tallest first. Colour still follows the model (never the rank), and every column is named
# and labelled with its value, so a reordering never leaves identity to colour alone.
import textwrap

def column_label(model_name):
    return textwrap.fill(display_name(model_name), 12, break_long_words=False)

# 3. Every question, every model: headline metric with its bootstrap 95% interval
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
for ax, question in zip(axes.flat, QUESTION_TITLES):
    metric = headline_names[QUESTIONS[question]["type"]]
    models = [model for model in CHART_MODELS if question in scores[model]]
    models = sorted(models, key=lambda model: -scores[model][question][metric])   # stable: ties keep the fixed order
    values = [scores[model][question][metric] for model in models]
    intervals = [tuple(float(bound) for bound in scores[model][question]["95% CI"].split("-")) for model in models]
    texture(ax.bar(range(len(models)), values, width=0.65, color=[MODEL_COLORS[model] for model in models]), models)
    for x, (value, (low, high)) in enumerate(zip(values, intervals)):
        ax.plot([x, x], [low, high], color=INK, linewidth=1)
        ax.text(x, max(high, value) + 0.02, f"{value:.2f}", ha="center", va="bottom", color=INK, fontsize=9)
    ax.set_xticks(range(len(models)), [display_name(model) for model in models], fontsize=8.5, rotation=35, ha="right", rotation_mode="anchor")
    ax.set_ylim(0, 1.12)
    ax.set_title(QUESTION_TITLES[question])
    ax.set_ylabel("Accuracy" if metric == "accuracy" else "F2 (catching the yes cases)")
    style_value_axis(ax, "y")
axes.flat[-1].axis("off")
axes.flat[-1].text(0, 0.6, "Columns: score on 1,000 test conversations, highest first\nLines: bootstrap 95% interval\n\nKeywords only answer the yes/no\nand resolution questions",
                   color=MUTED_INK, fontsize=10, va="top")
fig.suptitle("Zero-shot models vs baselines, question by question", x=0.01, ha="left", fontsize=15, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.975))
save_figure(fig, "03_scores_by_question")

# 4 and 5. Cost and latency per model (zero-shot models only; one measure per chart, no shared axis), highest first
for figure_name, column, title, label, formatter in [
    ("04_cost_per_1000", "cost per 1,000 conversations $", "Cost to label 1,000 conversations", "US dollars", lambda v: f"${v:,.2f}"),
    ("05_latency", "median latency s", "Median time per conversation (all 7 questions)", "Seconds", lambda v: f"{v:.2f} s"),
]:
    ordered = efficiency_table.loc[[model for model in ZERO_SHOT_CHART_MODELS if model in efficiency_table.index], column].astype(float)
    unpriced = list(ordered.index[ordered.isna()])
    if unpriced:
        print(f"{figure_name}: left out until a price or measurement exists: {unpriced}")
    ordered = ordered.dropna().sort_values(ascending=False, kind="stable")
    fig, ax = plt.subplots(figsize=(11, 4.8))
    texture(ax.bar(range(len(ordered)), ordered.values, width=0.65, color=[MODEL_COLORS[model] for model in ordered.index]), list(ordered.index))
    for x, value in enumerate(ordered.values):
        ax.text(x, value + ordered.max() * 0.015, formatter(value) + ("\n(own GPU)" if value == 0 and column.startswith("cost") else ""),
                ha="center", va="bottom", color=INK, fontsize=9)
    ax.set_xticks(range(len(ordered)), [column_label(model) for model in ordered.index])
    ax.set_ylim(0, ordered.max() * 1.15)
    ax.set_title(title)
    ax.set_ylabel(label)
    style_value_axis(ax, "y")
    save_figure(fig, figure_name)

# 7. Memorization probe: mean accuracy over the 7 questions, original vs identifying details swapped
probe_table = pd.DataFrame(probe_results).T
probe_models = [model for model in ZERO_SHOT_CHART_MODELS if model in probe_table.index][::-1]
fig, ax = plt.subplots(figsize=(8, 0.45 * len(probe_models) + 1.4))
for y, model in enumerate(probe_models):
    original, swapped = probe_table.loc[model, "mean accuracy, original"], probe_table.loc[model, "mean accuracy, details swapped"]
    ax.plot([original, swapped], [y, y], color=AXIS, linewidth=2, zorder=1)
    ax.scatter(original, y, s=70, facecolor=SURFACE, edgecolor=MODEL_COLORS[model], linewidth=2, zorder=2)
    ax.scatter(swapped, y, s=70, color=MODEL_COLORS[model], edgecolor=SURFACE, linewidth=2, zorder=3)
    change = swapped - original
    ax.text(max(original, swapped) + 0.012, y, "no change" if abs(change) < 0.0005 else f"{change:+.3f}", va="center", color=INK, fontsize=9)
ax.set_yticks(range(len(probe_models)), [display_name(model) for model in probe_models])
ax.set_xlim(0.5, 1.0)
ax.set_title("Memorization probe: accuracy before and after swapping names and IDs")
ax.set_xlabel(f"Mean accuracy over the 7 questions, the {len(changed_ids)} probe conversations that changed  (hollow = original, filled = swapped)")
style_value_axis(ax)
save_figure(fig, "07_memorization_probe")

# 8. Precision vs recall per yes/no label. Curves = models with probabilities (move the threshold, slide along the
# curve); markers = every model at its actual 0.5 / yes-no operating point. Marker shape doubles colour as identity.
MARKERS = {"Jev 1.13": "o", "Jev 1.14": "s", "GPT-6 Luna": "D", "Gemini 3.5 Flash-Lite": "^", "Claude Haiku 5.5": "P",
           "Clef": "h", "Clef-flash (Cloudflare)": "v", "Metis (GreenPT)": "<", "Clef-flash (local)": ">", SUPERVISED: "X", "Keywords": "*"}
CURVE_MODELS = [model for model in ["Jev 1.14", "Clef", "Clef-flash (Cloudflare)", SUPERVISED] if model in all_probabilities]
MARKER_SIZES = {"Keywords": 160}
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, question in zip(axes.flat, NOUL_QUESTIONS):
    truth = test[question].values.astype(bool)
    for model_name in CURVE_MODELS:
        precision, recall, _ = precision_recall_curve(truth, all_probabilities[model_name][question])
        ax.plot(recall, precision, color=MODEL_COLORS[model_name], linewidth=2, alpha=0.9, zorder=1)
    for model_name in CHART_MODELS:
        if question in scores[model_name]:
            point = scores[model_name][question]
            ax.scatter(point["recall"], point["precision"], s=MARKER_SIZES.get(model_name, 90), marker=MARKERS[model_name], color=MODEL_COLORS[model_name],
                       edgecolor=SURFACE, linewidth=1.5, zorder=3)
    ax.set_xlim(0, 1.03)
    ax.set_ylim(0, 1.03)
    ax.set_xlabel("Recall (share of real yes cases caught)")
    ax.set_ylabel("Precision (share of yes answers that were right)")
    ax.set_title(QUESTION_TITLES[question])
    ax.grid(color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
handles = [plt.Line2D([], [], color=MODEL_COLORS[model], linewidth=2, label=f"{display_name(model)}: threshold curve")
           for model in CURVE_MODELS]
handles += [plt.Line2D([], [], linestyle="", marker=MARKERS[model], markersize=9, color=MODEL_COLORS[model],
                       label=f"{display_name(model)}: its actual answers") for model in CHART_MODELS]
fig.suptitle("Precision vs recall: where each model lands, and what a moved threshold could buy", x=0.01, ha="left", fontsize=15, fontweight="bold")
fig.tight_layout(rect=(0, 0.09, 1, 0.97))
fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=9.5, bbox_to_anchor=(0.5, 0.0))
save_figure(fig, "08_precision_recall")

# 9. Latency: the distribution of every call, not just the median. Cumulative share of calls finished by time t.
latency_models = [model for model in ZERO_SHOT_CHART_MODELS if model in efficiency_table.index]
all_calls = {model: collect_calls for model, (_, _, collect_calls) in collect().items()}
fig, ax = plt.subplots(figsize=(10, 5.2))
for model_name in latency_models:
    latencies = np.sort(all_calls[model_name]["latency_s"].astype(float).values)
    share = np.arange(1, len(latencies) + 1) / len(latencies)
    median = np.median(latencies)
    ax.plot(latencies, share, color=MODEL_COLORS[model_name], linewidth=2, linestyle=MODEL_LINESTYLES.get(model_name, "-"),
            label=f"{display_name(model_name)}  (median {median:.2f} s)")
ax.set_xscale("log")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda value, _: f"{value:g} s"))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
ax.set_xlabel("Time to label one conversation, all 7 questions (log scale)")
ax.set_ylabel("Share of the 1,000 calls finished")
ax.set_title("How long each call takes, every call")
handles, labels = ax.get_legend_handles_labels()
order = np.argsort([float(label.split("median ")[1].split(" s")[0]) for label in labels])   # fastest first, matching left to right
ax.legend([handles[i] for i in order], [labels[i] for i in order], loc="lower right", fontsize=9.5)
ax.grid(color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
save_figure(fig, "09_latency_distribution")

# 10. Routing by call reason: share of each reason's calls labelled with the right reason (per-class recall)
routing_models = [model for model in CHART_MODELS if "reason" in scores[model]]
reasons = list(FLOW_DESCRIPTIONS)
routing = pd.DataFrame({model: [np.mean(all_predictions[model]["reason"][test["reason"].values == reason] == reason) for reason in reasons]
                        for model in routing_models}, index=reasons).T
fig, ax = plt.subplots(figsize=(13, 0.5 * len(routing_models) + 1.5))
# Diverging red <-> blue from the reference palette, neutral grey at 0.5: misrouting reads as red, good routing as blue
from matplotlib.colors import LinearSegmentedColormap
routing_cmap = LinearSegmentedColormap.from_list("routing", ["#b52a2a", "#e34948", "#f0efec", "#2a78d6", "#1a4f94"])
image = ax.imshow(routing.values, cmap=routing_cmap, vmin=0, vmax=1, aspect="auto")
for (row, col), value in np.ndenumerate(routing.values):
    ax.text(col, row, f"{value:.2f}", ha="center", va="center", fontsize=9, color=SURFACE if abs(value - 0.5) > 0.24 else INK)
ax.set_xticks(range(len(reasons)), [textwrap.fill(reason.replace("_", " "), 12, break_long_words=False) for reason in reasons], fontsize=9)
ax.set_yticks(range(len(routing_models)), [display_name(model) for model in routing_models])
ax.set_title("Routing by call reason: share of each reason's calls sent to the right place")
for spine in ax.spines.values():
    spine.set_visible(False)
fig.colorbar(image, ax=ax, fraction=0.025, pad=0.01).outline.set_visible(False)
save_figure(fig, "10_reason_routing")

# 11. The identity fix: false alarms and misses per model, round one (hollow) to round two (filled)
if identity_rows:
    fix_models = [model for model in ZERO_SHOT_CHART_MODELS if model in identity_before_after.index][::-1]
    fig, axes = plt.subplots(1, 2, figsize=(13, 0.55 * len(fix_models) + 1.6), sharey=True)
    for ax, measure, title in [(axes[0], "false alarms", "False alarms (said verified, it wasn't)"),
                               (axes[1], "misses", "Misses (verified, said it wasn't)")]:
        for y, model in enumerate(fix_models):
            before, after = identity_before_after.loc[model, f"{measure}, round one"], identity_before_after.loc[model, f"{measure}, round two"]
            ax.plot([before, after], [y, y], color=AXIS, linewidth=2, zorder=1)
            ax.scatter(before, y, s=70, facecolor=SURFACE, edgecolor=MODEL_COLORS[model], linewidth=2, zorder=2)
            ax.scatter(after, y, s=70, color=MODEL_COLORS[model], edgecolor=SURFACE, linewidth=2, zorder=3)
        x_max = max(identity_before_after[[f"{measure}, round one", f"{measure}, round two"]].values.max() * 1.25, 10)
        for y, model in enumerate(fix_models):
            before, after = identity_before_after.loc[model, f"{measure}, round one"], identity_before_after.loc[model, f"{measure}, round two"]
            ax.text(max(before, after) + x_max * 0.02, y, f"{before:.0f} → {after:.0f}", va="center", color=INK, fontsize=9)
        ax.set_yticks(range(len(fix_models)), [display_name(model) for model in fix_models])
        ax.set_xlim(0, x_max)
        ax.set_title(title)
        ax.set_xlabel("Conversations, of 1,000  (hollow = round one wording, filled = round two)")
        style_value_axis(ax)
    fig.suptitle("One definition, before and after: the identity question", x=0.01, ha="left", fontsize=15, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    save_figure(fig, "11_identity_before_after")

# 12. Same weights, three hosts: label agreement per question and host pair (one sequential hue: higher = darker)
if host_rows:
    agreement = host_table.pivot(index="pair", columns="question", values="label agreement")[list(QUESTION_TITLES)]
    fig, ax = plt.subplots(figsize=(13, 0.7 * len(agreement) + 1.6))
    agreement_cmap = LinearSegmentedColormap.from_list("agreement", ["#f0efec", "#86b6ef", "#2a78d6", "#104281"])
    low = min(0.9, float(np.floor(agreement.values.min() * 50) / 50))
    image = ax.imshow(agreement.values, cmap=agreement_cmap, vmin=low, vmax=1, aspect="auto")
    for (row, col), value in np.ndenumerate(agreement.values):
        ax.text(col, row, f"{value:.1%}", ha="center", va="center", fontsize=9.5,
                color=SURFACE if (value - low) / (1 - low) > 0.55 else INK)
    ax.set_xticks(range(agreement.shape[1]), [textwrap.fill(QUESTION_TITLES[q], 14) for q in agreement.columns], fontsize=9)
    ax.set_yticks(range(len(agreement)), [pair.replace(" vs ", "\nvs ") for pair in agreement.index], fontsize=9)
    ax.set_title("Same weights, three hosts: how often the labels agree")
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.colorbar(image, ax=ax, fraction=0.025, pad=0.01, format=mticker.PercentFormatter(1.0, decimals=0)).outline.set_visible(False)
    save_figure(fig, "12_three_hosts_agreement")

## Caveats

Written before the run; the results-dependent ones are filled in after scoring, as in round one.

**Design caveats (hold whatever the results)**
- **The freeze rule was changed after seeing dev results.** As first written ("best mean identity F2"), it picked round one's wording `v0` over `v3` by 0.001 mean F2 on 300 dev chats. On 2026-10-20, after seeing 6 of 8 models' dev results and before any test call, a tie-break was added: within 0.005 F2 is a tie, and ties go to fewer total errors, which picks `v3` (about 38 errors per model vs 98 for `v0` on dev). The change was made knowing which version it favours. Once Clef and Clef-flash finished dev, `v3` also had the best mean F2 outright across all 8 API models (0.830 vs 0.827), so the original rule picks it as well. To keep the choice checkable, `v0` and the reader's `v2` run on test too (section 13a).
- **Role-played data and system-event labels**, as in round one. The yes/no labels measure logged system events, not everything a person would call verification, a refund or a handoff.
- **The new identity wording was written by reading the dataset's action definitions** (which slots `verify-identity` takes), then iterated on 300 dev conversations. It describes the logged event precisely; that's the point of the round, and also why it isn't a blind definition. The test split was never used to write or choose it.
- **One question changed, one call answers all seven.** Before/after differences on the other six questions mix the wording's effect with ordinary run-to-run variation of the cloud LLMs; the repeat control sizes the second.
- **Round one's answers are from round one's calls** (2026-09/10). Providers can change a model behind an unchanged ID between rounds; the logged `served_model` is the only check.
- **Metis's underlying model can change without the ID changing** (GreenPT's docs). Section 12 checks its impact version and served model for the whole run.
- **Three hosts are not three identical setups.** Metis is full precision per GreenPT; Cloudflare doesn't state its precision; the local copy is quantized to fit 16 GB. Differences involving the local copy measure quantization, not hosting.
- **Energy figures aren't comparable to each other.** GreenPT reports its own per-call energy and emissions (its method, its units); the local figure is the whole GPU's energy counter around each call, idle draw and anything else on the GPU included; Cloudflare reports none. A ~40 ms call can be close to the counter's resolution.
- **Prices are list prices on logged tokens**, checked 2026-10-19. Workers AI bills in neurons (the per-token prices are Cloudflare's conversion) with 10,000 free neurons a day, not subtracted. Metis is billed in EUR and converted at one day's ECB rate, so its dollar cost moves with the exchange rate. Each host counts tokens with its own tokenizer (2,684 per call on Cloudflare and GreenPT, 2,378 on Jev in the smoke test).
- **Latency is one sequential call at a time** from one home connection (and from wherever the Metis calls run, if not from home). The local model has no network in its latency.
- **Keyword regexes were written after looking at the whole dataset**, test included (round one); the keyword baseline is exploratory.

**To fill in after the run**
- The fix: identity false alarms and misses before vs after, per model, and whether any model got worse on misses.
- The collateral check: any untouched question that moved significantly.
- Jev 1.13 vs 1.14, and where Clef and Clef-flash land against the LLMs.
- The three hosts: agreement, probability gaps, any significant score difference.
- Probe flips against repeat flips, per model.
- Reasoning tokens and cache hit shares at provider defaults (as in round one).